# Cyclone Oceanic Response Analysis: Scale Decomposition

**Cyclones Analyzed:** Remal (May 2024) & Dana (October 2024)

**Data Sources:**
- SST: GHRSST MUR-JPL-L4-GLOB-v4.1 (0.01° resolution)
- MSLA: CMEMS SEALEVEL_GLO_PHY_L4_MY_008_047 (0.125Â° resolution)

**Phases:**
1. Date-wise Data Inventory & First-look Plots
2. 2D Gaussian Filter for Scale Separation
3. Scale-wise Visual Comparison
4. Cyclone Response by Scale 

In [ ]:
import os
import glob
import re
import json
import gc
import warnings
from datetime import datetime, timedelta
from math import ceil
from calendar import month_name

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr
import cmocean

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150})

print('Libraries loaded successfully!')


In [ ]:
# =============================================================================
# CONFIGURATION  --  Edit ONLY this section
# =============================================================================

# ---- Year -------------------------------------------------------------------
# Change YEAR to process a different season.
# Also update MSLA_ROOT_DIR and CYCLONE_JSON_PATH below when changing year.
YEAR = 2024

# ---- Paths ------------------------------------------------------------------
SST_DATA_DIR      = r"user_path_for_sst"
MSLA_ROOT_DIR     = r"user_path_for_msla"
OUTPUT_DIR        = r"user_path_for_output"
CYCLONE_JSON_PATH = r"user_path_for_cyclone_json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Cyclone filter ---------------------------------------------------------
SELECT_BASINS = None   # e.g. ['bay_of_bengal']  or None for all basins
SELECT_STORMS = None   # e.g. ['REMAL', 'DANA']  or None for all storms

# ---- Analysis window padding ------------------------------------------------
PRE_DAYS  = None   # int or None -- extra days before formation
POST_DAYS = None   # int or None -- extra days after dissipation/landfall

# ---- Raw-field colour limits ------------------------------------------------
SST_VMIN, SST_VMAX = 28.0, 35.0   # degrees C
SLA_VMIN, SLA_VMAX = -0.2,  0.2   # metres

# ---- Gaussian filter scales (degrees) --------------------------------------
FILTER_SCALES = {
    'submeso': 0.25,   # ~28 km
    'meso':    1.0,    # ~111 km
    'large':   3.0,    # ~333 km
}
FILTER_SCALES_MSLA = {
    'submeso': 0.625,  # ~69 km
    'meso':    1.5,    # ~167 km
    'large':   3.0,    # ~333 km
}

# ---- Colormaps --------------------------------------------------------------
CMAP_SST      = plt.get_cmap('turbo')
CMAP_SLA      = plt.get_cmap('RdBu_r')
CMAP_SST_ANOM = plt.get_cmap('RdBu_r')
CMAP_SLA_ANOM = plt.get_cmap('RdBu_r')
CMAP_GRADIENT = plt.get_cmap('magma')   # gradient-magnitude panels

# ---- Panel style ------------------------------------------------------------
ANALYSIS_COLOR = '#444444'
ANALYSIS_TEXT  = 'white'

# ---- Scale-decomposition panel rows (evolution matrix) ---------------------
SCALE_ROWS = [
    {
        'key':   'raw',
        'label': 'Raw Field',
        'sst':  {'cmap': CMAP_SST,      'vmin': SST_VMIN, 'vmax': SST_VMAX, 'unit': 'C',            'diverge': False},
        'msla': {'cmap': CMAP_SLA,      'vmin': SLA_VMIN, 'vmax': SLA_VMAX, 'unit': 'm',             'diverge': True},
    },
    {
        'key':   'large_scale',
        'label': 'Large-Scale Anomaly\n(>333 km SST  |  >333 km MSLA)',
        'sst':  {'cmap': CMAP_SST_ANOM, 'vmin': -0.3,  'vmax':  0.3,  'unit': 'SST Anom (C)',  'diverge': True},
        'msla': {'cmap': CMAP_SLA_ANOM, 'vmin': -0.12, 'vmax':  0.12, 'unit': 'SLA Anom (m)', 'diverge': True},
    },
    {
        'key':   'meso',
        'label': 'Mesoscale Band\n(111-333 km SST  |  167-333 km MSLA)',
        'sst':  {'cmap': CMAP_SST_ANOM, 'vmin': -0.5,  'vmax':  0.5,  'unit': 'SST Anom (C)',  'diverge': True},
        'msla': {'cmap': CMAP_SLA_ANOM, 'vmin': -0.10, 'vmax':  0.10, 'unit': 'SLA Anom (m)', 'diverge': True},
    },
    {
        'key':   'submeso',
        'label': 'Sub-Mesoscale\n(<111 km SST  |  <167 km MSLA)',
        'sst':  {'cmap': CMAP_SST_ANOM, 'vmin': -0.3,  'vmax':  0.3,  'unit': 'SST Anom (C)',  'diverge': True},
        'msla': {'cmap': CMAP_SLA_ANOM, 'vmin': -0.04, 'vmax':  0.04, 'unit': 'SLA Anom (m)', 'diverge': True},
    },
]

# ---- Gradient-magnitude panel rows (gradient matrix) -----------------------
SCALE_GRAD_ROWS = [
    {
        'key':   'raw',
        'label': 'Raw Field',
        'sst':  {'cmap': CMAP_SST,      'vmin': SST_VMIN, 'vmax': SST_VMAX, 'unit': 'C',       'diverge': False},
        'msla': {'cmap': CMAP_SLA,      'vmin': SLA_VMIN, 'vmax': SLA_VMAX, 'unit': 'm',        'diverge': True},
    },
    {
        'key':   'large_grad',
        'label': 'Large-Scale Grad\nMagnitude',
        'sst':  {'cmap': CMAP_GRADIENT, 'vmin': 0.0, 'vmax': 0.005, 'unit': 'C / km',  'diverge': False},
        'msla': {'cmap': CMAP_GRADIENT, 'vmin': 0.0, 'vmax': 0.001, 'unit': 'm / km',  'diverge': False},
    },
    {
        'key':   'meso_grad',
        'label': 'Meso-Scale Grad\nMagnitude',
        'sst':  {'cmap': CMAP_GRADIENT, 'vmin': 0.0, 'vmax': 0.03,  'unit': 'C / km',  'diverge': False},
        'msla': {'cmap': CMAP_GRADIENT, 'vmin': 0.0, 'vmax': 0.002, 'unit': 'm / km',  'diverge': False},
    },
    {
        'key':   'submeso_grad',
        'label': 'Submeso-Scale Grad\nMagnitude',
        'sst':  {'cmap': CMAP_GRADIENT, 'vmin': 0.0, 'vmax': 0.05,  'unit': 'C / km',  'diverge': False},
        'msla': {'cmap': CMAP_GRADIENT, 'vmin': 0.0, 'vmax': 0.003, 'unit': 'm / km',  'diverge': False},
    },
]


# =============================================================================
# CYCLONE DATA  --  Helper functions (do not edit below)
# =============================================================================

def _parse_date(date_str):
    '''
    Parse an ISO date string into a datetime object.

    Parameters
    ----------
    date_str : str or None
        Date in YYYY-MM-DD format, or None.

    Returns
    -------
    datetime.datetime or None
        Parsed datetime at midnight, or None when date_str is falsy.
    '''
    return datetime.strptime(date_str, "%Y-%m-%d") if date_str else None


def _mph_to_kt(mph):
    '''
    Convert wind speed from miles per hour to knots.

    Parameters
    ----------
    mph : float or None

    Returns
    -------
    float or None
        Rounded to one decimal place, or None.
    '''
    return round(mph * 0.868976, 1) if mph is not None else None


def _derive_analysis_dates(dates):
    '''
    Derive pre / during / post phase windows from a cyclone dates dict.

    The during phase spans formation to landfall (or dissipation).  Pre and
    post phases extend by PRE_DAYS and POST_DAYS respectively.

    Parameters
    ----------
    dates : dict
        Keys formation and dissipation (required), optionally landfall.

    Returns
    -------
    dict
        Keys pre, during, post each a 2-tuple (start_str, end_str),
        or empty dict when formation or dissipation is missing.
    '''
    formation   = _parse_date(dates.get("formation"))
    dissipation = _parse_date(dates.get("dissipation"))
    landfall    = _parse_date(dates.get("landfall"))

    if not formation or not dissipation:
        return {}

    during_end = landfall or dissipation
    pre_days   = PRE_DAYS  or 0
    post_days  = POST_DAYS or 0

    return {
        "pre": (
            (formation - timedelta(days=pre_days)).strftime("%Y-%m-%d"),
            (formation - timedelta(days=1)).strftime("%Y-%m-%d"),
        ),
        "during": (
            formation.strftime("%Y-%m-%d"),
            during_end.strftime("%Y-%m-%d"),
        ),
        "post": (
            (during_end + timedelta(days=1)).strftime("%Y-%m-%d"),
            (during_end + timedelta(days=post_days)).strftime("%Y-%m-%d"),
        ),
    }


def _normalize_storm_record(storm):
    '''
    Convert a raw JSON storm dict into the standard internal representation.

    Applies unit conversion (mph to kt) and derives analysis date windows
    via _derive_analysis_dates if not already stored.

    Parameters
    ----------
    storm : dict
        A single storm entry as read from cyclone_info_padded.json.

    Returns
    -------
    dict
        Normalised record with keys: name, dates, analysis_dates,
        max_wind_kt, min_pressure_hPa plus optional track, bbox, category.
    '''
    max_wind_kt  = storm.get("max_wind_kt") or _mph_to_kt(storm.get("max_wind_mph"))
    min_pressure = storm.get("min_pressure_hPa") or storm.get("min_pressure_mb")
    dates          = storm.get("dates", {})
    analysis_dates = storm.get("analysis_dates") or _derive_analysis_dates(dates)
    normalized = {
        "name"            : storm.get("name", "Unknown"),
        "dates"           : dates,
        "analysis_dates"  : analysis_dates,
        "max_wind_kt"     : max_wind_kt,
        "min_pressure_hPa": min_pressure,
    }
    if "track"    in storm: normalized["track"]    = storm["track"]
    if "bbox"     in storm: normalized["bbox"]     = storm["bbox"]
    if "category" in storm: normalized["category"] = storm["category"]
    return normalized


def load_cyclones(json_path):
    '''
    Load and normalise all cyclone records from the JSON database.

    Reads the JSON file, iterates every basin and storm, applies
    SELECT_BASINS / SELECT_STORMS filters, and normalises each record.

    Parameters
    ----------
    json_path : str
        Path to the padded cyclone JSON file.

    Returns
    -------
    dict
        STORM_NAME_UPPER: normalised_record for every passing storm.
    '''
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    cyclones = {}
    for basin_key, basin in data.items():
        if SELECT_BASINS and basin_key.lower() not in [b.lower() for b in SELECT_BASINS]:
            continue
        for storm in basin.get("storms", []):
            name = storm.get("name", "").strip()
            if SELECT_STORMS and name.lower() not in [n.lower() for n in SELECT_STORMS]:
                continue
            cyclones[name.upper()] = _normalize_storm_record(storm)
    return cyclones


# =============================================================================
# MAIN  --  Load cyclones from JSON
# =============================================================================

cyclones = load_cyclones(CYCLONE_JSON_PATH)

# results = {}
# for name, info in cyclones.items():
#     formation   = info["dates"].get("formation")
#     dissipation = info["dates"].get("dissipation")
#     if not formation or not dissipation:
#         continue
#     start = _parse_date(formation)
#     end   = _parse_date(dissipation)
#     days  = (end - start).days + 1
#     results[name] = days
# filtered = {k: v for k, v in results.items() if v >= 6}
# for name, days in sorted(filtered.items(), key=lambda x: x[1], reverse=True):
#     print(f"{name}: {days} days")
# print("Total cyclones:", len(results))
# print("Cyclones >=6 days:", len(filtered))


In [ ]:
# import json
# from datetime import datetime, timedelta

# OUTPUT_PATH = r"C:\Users\abhik\Desktop\project related work\cyclone_info_2024_padded.json"


# =========================
# HELPERS
# =========================

# def _format(date_value):
#     if isinstance(date_value, datetime):
#         return date_value.strftime("%Y-%m-%d")
#     return datetime.strptime(date_value, "%Y-%m-%d").strftime("%Y-%m-%d")


# def get_padding(days, storm_name=None):
#     """Return (pre_days, post_days) to normalize durations to 12 or 20 days."""
#     name = (storm_name or "").strip().upper()
#     if name == "ANGGREK" and days == 22:
#         return (0, 0)
#     if days == 20 or days == 12:
#         return (0, 0)
#     if 12 < days < 20:
#         needed = 20 - days
#         pre = needed // 2
#         post = needed - pre
#         return (pre, post)
#     else:
#         needed = 12 - days
#         pre = needed // 2
#         post = needed - pre
#         return (pre, post)


# =========================
# MAIN PROCESS
# =========================
# with open(CYCLONE_JSON_PATH, "r", encoding="utf-8") as f:
#     data = json.load(f)


# for basin in data.values():
#     for storm in basin.get("storms", []):
#         dates = storm.get("dates", {})

#         formation = dates.get("formation")
#         dissipation = dates.get("dissipation")

#         if not formation or not dissipation:
#             continue

#         start = _parse_date(formation)
#         end = _parse_date(dissipation)

#         days = (end - start).days + 1

#         pre_days, post_days = get_padding(days, storm.get("name"))

#         start_analysis = start - timedelta(days=pre_days)
#         end_analysis = end + timedelta(days=post_days)

#         # Keep formation/dissipation unchanged; write analysis window separately
#         dates["start_analysis_date"] = _format(start_analysis)
#         dates["end_analysis_date"] = _format(end_analysis)


# =========================
# SAVE UPDATED FILE
# =========================
# with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#     json.dump(data, f, indent=2)


# print("✅ JSON updated with analysis window dates (12 or 20 days)")
# name_days = []

# for basin in data.values():
#     for storm in basin.get("storms", []):
#         name = storm.get("name")
#         dates = storm.get("dates", {})
#         formation = dates.get("formation")
#         dissipation = dates.get("dissipation")
#         start_analysis = dates.get("start_analysis_date")
#         end_analysis = dates.get("end_analysis_date")
#         if not name or not formation or not dissipation:
#             continue
#         start = datetime.strptime(formation, "%Y-%m-%d")
#         end = datetime.strptime(dissipation, "%Y-%m-%d")
#         days = (end - start).days + 1
#         start = datetime.strptime(start_analysis, "%Y-%m-%d")
#         end = datetime.strptime(end_analysis, "%Y-%m-%d")
#         pad_days = (end - start).days + 1
#         name_days.append((name, days, pad_days))

# print("Cyclone durations (days):")
# for name, days, pad_days in sorted(name_days, key=lambda x: x[0].lower()):
#     print(f"{name}: {days} (padded: {pad_days})")

---
# PHASE 1: Date-wise Data Inventory & First-look Plots

## 1.1 Cyclone Track Summary

In [ ]:
def print_cyclone_summary(cyclone_data=None):
    '''
    Print a formatted inventory summary for every cyclone in the dataset.

    Handles both the normalised flat structure from load_cyclones and the
    original basin-grouped JSON structure.

    Parameters
    ----------
    cyclone_data : dict or None, optional
        Cyclone records to summarise.  When None uses the module-level
        cyclones dict.

    Returns
    -------
    None
        Prints directly to stdout.
    '''
    data = cyclone_data if cyclone_data is not None else cyclones
    print("=" * 80)
    print("CYCLONE INVENTORY SUMMARY")
    print("=" * 80)

    def _derive_bbox_from_track(track):
        if not track:
            return None
        lons = [p[0] for p in track]
        lats = [p[1] for p in track]
        return {"lon_min": min(lons), "lon_max": max(lons),
                "lat_min": min(lats), "lat_max": max(lats)}

    def _print_storm(cyc_name, cyc):
        print(f"\n{'=' * 40}")
        print(f"CYCLONE: {cyc.get('name', cyc_name)}")
        print(f"{'=' * 40}")
        print(f"Category: {cyc.get('category', 'N/A')}")
        max_wind  = cyc.get('max_wind_mph') or cyc.get('max_wind_kt', 'N/A')
        min_press = cyc.get('min_pressure_mb') or cyc.get('min_pressure_hPa', 'N/A')
        print(f"Max Wind: {max_wind} | Min Pressure: {min_press}")
        print(f"\nKEY DATES:")
        for key, date in cyc.get('dates', {}).items():
            print(f"  {key.capitalize():20} : {date}")
        track = cyc.get('track', [])
        bb = cyc.get('bbox') or _derive_bbox_from_track(track)
        print(f"\nANALYSIS BOUNDING BOX:")
        if bb:
            print(f"  Longitude: {bb.get('lon_min')} E to {bb.get('lon_max')} E")
            print(f"  Latitude:  {bb.get('lat_min')} N to {bb.get('lat_max')} N")
        else:
            print("  N/A")
        analysis_dates = cyc.get('analysis_dates', {})
        if analysis_dates:
            print(f"\nANALYSIS PERIODS:")
            for phase, dates in analysis_dates.items():
                if isinstance(dates, (list, tuple)) and len(dates) == 2:
                    print(f"  {phase.capitalize():8} : {dates[0]} to {dates[1]}")
                else:
                    print(f"  {phase.capitalize():8} : {dates}")
        print(f"\nTRACK POSITIONS ({len(track)} points):")
        print(f"  {'Date/Time':<18} {'Lon':>8} {'Lat':>8} {'Wind':>8} {'Pres':>8}")
        for point in track:
            if len(point) == 5:
                lon, lat, dt, wind, pres = point
                w_str = f"{wind:>6}" if wind is not None else "   N/A"
                p_str = f"{pres:>6}" if pres is not None else "   N/A"
                print(f"  {dt:<18} {float(lon):>8.1f} {float(lat):>8.1f} {w_str} {p_str}")
            else:
                print(f"  {str(point)}")

    if any(isinstance(v, dict) and 'storms' in v for v in data.values()):
        for basin in data.values():
            for storm in basin.get('storms', []):
                _print_storm(storm.get('name', 'Unknown'), storm)
    else:
        for cyc_name, cyc in data.items():
            _print_storm(cyc_name, cyc)

    print(f"\n{'=' * 80}")

# print_cyclone_summary()


## 1.2 Data Loading Functions

In [ ]:
def extract_date_from_sst_filename(filepath):
    '''
    Parse the observation date from a MUR-JPL SST NetCDF filename.

    Filenames start with YYYYMMDDHHMMSS (e.g. 20240526090000-JPL-...).
    Only the leading 8-digit YYYYMMDD portion is extracted.

    Parameters
    ----------
    filepath : str

    Returns
    -------
    datetime.datetime

    Raises
    ------
    ValueError
        If the filename does not start with an 8-digit date token.
    '''
    filename = os.path.basename(filepath)
    match = re.match(r'^(\\d{8})', filename)
    if match:
        return datetime.strptime(match.group(1), '%Y%m%d')
    raise ValueError(f"Cannot parse date from: {filename}")


def load_sst_for_date(date, region, downsample=5):
    '''
    Load SST for one calendar date and spatial region from MUR-JPL files.

    Parameters
    ----------
    date : datetime.datetime or str
        Target date (YYYY-MM-DD if string).
    region : dict
        Bounding box: lon_min, lon_max, lat_min, lat_max.
    downsample : int, optional
        Stride for both lat/lon axes.  Default 5 gives 0.05 deg grid.

    Returns
    -------
    xr.DataArray or None
        2-D SST in degrees Celsius, or None if no file found.
    '''
    if isinstance(date, str):
        date = datetime.strptime(date, '%Y-%m-%d')
    date_str = date.strftime('%Y%m%d')
    pattern  = os.path.join(SST_DATA_DIR, f"{date_str}*.nc")
    files    = glob.glob(pattern)
    if not files:
        return None
    ds  = xr.open_dataset(files[0])
    sst = ds['analysed_sst'].squeeze('time')
    sst = sst.sel(lon=slice(region['lon_min'], region['lon_max']),
                  lat=slice(region['lat_min'], region['lat_max']))
    sst = sst - 273.15
    if downsample > 1:
        sst = sst[::downsample, ::downsample]
    ds.close()
    return sst


def extract_date_from_msla_filename(filepath):
    '''
    Parse the observation date from a CMEMS DUACS MSLA NetCDF filename.

    Filenames embed the data date as _YYYYMMDD_ (e.g.
    dt_global_allsat_phy_l4_20240526_20250429.nc).

    Parameters
    ----------
    filepath : str

    Returns
    -------
    datetime.datetime

    Raises
    ------
    ValueError
        If no 8-digit date token is found in the filename.
    '''
    filename = os.path.basename(filepath)
    match = re.search(r'_(\\d{8})_', filename)
    if match:
        return datetime.strptime(match.group(1), '%Y%m%d')
    raise ValueError(f"Cannot parse date from: {filename}")


def load_msla_for_date(date, region, downsample=1):
    '''
    Load MSLA (SLA) for one calendar date and spatial region from CMEMS files.

    Parameters
    ----------
    date : datetime.datetime or str
        Target date (YYYY-MM-DD if string).
    region : dict
        Bounding box: lon_min, lon_max, lat_min, lat_max.
    downsample : int, optional
        Stride for both lat/lon axes.  Default 1 (no downsampling).

    Returns
    -------
    xr.DataArray or None
        2-D SLA in metres, or None if no file found.
    '''
    if isinstance(date, str):
        date = datetime.strptime(date, '%Y-%m-%d')
    date_str = date.strftime('%Y%m%d')
    pattern  = os.path.join(MSLA_ROOT_DIR, f"*_{date_str}_*.nc")
    files    = glob.glob(pattern)
    if not files:
        return None
    ds   = xr.open_dataset(files[0], engine="netcdf4")
    msla = ds['sla'].squeeze('time')
    msla = msla.sel(latitude=slice(region['lat_min'],  region['lat_max']),
                    longitude=slice(region['lon_min'], region['lon_max']))
    if downsample > 1:
        msla = msla[::downsample, ::downsample]
    ds.close()
    return msla


print("Data loading functions defined.")


## 1.3 Load MSLA Dataset

In [ ]:

# ---------------------------------------------------------
# ON-DEMAND APPROACH (MATCHES SST): Do not pre-load MSLA.
# ---------------------------------------------------------

# 1) Pre-calculate the bounding boxes and analysis dates for all cyclones
with open(CYCLONE_JSON_PATH, "r", encoding="utf-8") as f:
    json_data = json.load(f)

cyc_bboxes = {}
for basin_key, basin_info in json_data.items():
    for storm in basin_info.get("storms", []):
        cyc_name = storm.get("name", "Unknown")
        bb = storm.get("bbox")
        dates = storm.get("dates", {})
        
        start_str = dates.get("start_analysis_date")
        end_str = dates.get("end_analysis_date")
        
        if not bb:
            track = storm.get("track", [])
            if track:
                lons = [float(p[0]) for p in track]
                lats = [float(p[1]) for p in track]
                bb = {
                    "lon_min": min(lons), "lon_max": max(lons),
                    "lat_min": min(lats), "lat_max": max(lats)
                }
            else:
                continue
                
        if not start_str or not end_str:
            continue
            
        start_date = datetime.strptime(start_str, "%Y-%m-%d").date()
        end_date = datetime.strptime(end_str, "%Y-%m-%d").date()
                
        # Buffer
        cyc_bboxes[cyc_name] = {
            'lat_min': bb['lat_min'] - 1, 'lat_max': bb['lat_max'] + 1,
            'lon_min': bb['lon_min'] - 1, 'lon_max': bb['lon_max'] + 1,
            'start_date': start_date,
            'end_date': end_date
        }

print(f"Prepared bounding boxes and dates for {len(cyc_bboxes)} cyclones.")

## 1.4 Data Inventory - Check Available Files

In [ ]:
from datetime import timedelta
import pandas as pd
import os
import glob

def check_data_availability_all_cyclones():
    '''
    Report daily SST and MSLA file availability for every cyclone in cyc_bboxes.

    For each cyclone iterates over every calendar day in the analysis window,
    checks whether matching SST and MSLA files exist, and prints a day-by-day
    summary table.

    Returns
    -------
    dict
        {cyclone_name: {dates, sst_count, msla_count, total_days}}

    Notes
    -----
    Relies on SST_DATA_DIR and MSLA_ROOT_DIR from the Configuration cell.
    Call after cyc_bboxes has been populated.
    '''"""

    results = {}

    msla_files = glob.glob(os.path.join(MSLA_ROOT_DIR, "*.nc"))
    if not msla_files:
        print(f"WARNING: No MSLA files found in {MSLA_ROOT_DIR}")

    msla_dates = set()
    for fp in msla_files:
        try:
            msla_dates.add(extract_date_from_msla_filename(fp).date())
        except Exception:
            continue

    for cyclone_name in sorted(cyc_bboxes.keys()):
        cyc_info = cyc_bboxes[cyclone_name]
        start_dt = datetime.combine(cyc_info['start_date'], datetime.min.time())
        end_dt = datetime.combine(cyc_info['end_date'], datetime.min.time())

        print(f"\n{'=' * 60}")
        print(f"DATA AVAILABILITY: {cyclone_name}")
        print(f"{'=' * 60}")

        all_dates = []
        current = start_dt
        while current <= end_dt:
            all_dates.append((current, 'Analysis'))
            current += timedelta(days=1)

        print(f"\n{'Date':<12} {'Phase':<10} {'SST':<8} {'MSLA':<8}")
        print("-" * 40)

        sst_count = 0
        msla_count = 0

        for date, phase in all_dates:
            date_str = date.strftime('%Y%m%d')
            sst_pattern = os.path.join(SST_DATA_DIR, f"{date_str}*.nc")
            sst_files = glob.glob(sst_pattern)
            sst_ok = len(sst_files) > 0

            msla_ok = date.date() in msla_dates

            if sst_ok:
                sst_count += 1
            if msla_ok:
                msla_count += 1

            sst_status = '✓' if sst_ok else '✗'
            msla_status = '✓' if msla_ok else '✗'
            print(f"{date.strftime('%Y-%m-%d'):<12} {phase:<10} {sst_status:<8} {msla_status:<8}")

        print("-" * 40)
        print(f"Total: {len(all_dates)} days | SST: {sst_count} | MSLA: {msla_count}")

        results[cyclone_name] = {
            "dates": all_dates,
            "sst_count": sst_count,
            "msla_count": msla_count,
            "total_days": len(all_dates),
        }

    return results


# availability_results = check_data_availability_all_cyclones()

## 1.5 Cyclone Track Visualization

In [ ]:
def make_safe_name(name):
    '''
    Convert a cyclone name string into a Windows-safe file/folder name.

    Converts to uppercase, replaces Windows-reserved characters with
    underscores, collapses whitespace, and strips trailing periods/spaces.

    Parameters
    ----------
    name : str

    Returns
    -------
    str
        Sanitised uppercase name safe for use as a path component.
    '''
    name = str(name).strip().upper()
    name = re.sub(r'[<>:"/\\\\|?*]+', '_', name)
    name = re.sub(r'\\s+', ' ', name).strip()
    name = name.rstrip('. ')
    return name


CYCLONE_OUTPUT_DIRS = {}


def get_cyclone_output_dir(cyclone_name):
    '''
    Return (and create if necessary) the per-cyclone output directory.

    Normalises the name, constructs a path under OUTPUT_DIR, creates the
    directory, and caches the result.

    Parameters
    ----------
    cyclone_name : str
        Cyclone name as stored in the cyclones dict keys (uppercase).

    Returns
    -------
    str
        Absolute path to the cyclone-specific output directory.
    '''
    cyclone_key = str(cyclone_name).strip().upper()
    if cyclone_key not in CYCLONE_OUTPUT_DIRS:
        safe_name = make_safe_name(cyclone_key)
        cyc_dir   = os.path.join(OUTPUT_DIR, safe_name)
        os.makedirs(cyc_dir, exist_ok=True)
        CYCLONE_OUTPUT_DIRS[cyclone_key] = cyc_dir
    return CYCLONE_OUTPUT_DIRS[cyclone_key]


In [ ]:
def plot_individual_cyclone_tracks():
    '''
    Plot and save one individual track map per cyclone.

    Creates a separate figure per storm showing the track coloured by the
    per-cyclone colour with markers scaled by wind speed.  Key positions
    (first, last, peak-wind) are annotated.

    Returns
    -------
    list of tuple
        [(cyclone_name, saved_filepath), ...]
    '''
    color_list = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'magenta', 'cyan']
    marker_list = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']

    cyclone_names = sorted(cyclones.keys())
    colors = {name: color_list[i % len(color_list)] for i, name in enumerate(cyclone_names)}
    markers = {name: marker_list[i % len(marker_list)] for i, name in enumerate(cyclone_names)}

    saved_files = []

    for cyc_name, cyc in cyclones.items():
        if 'track' not in cyc or not cyc['track']:
            print(f"Skipping {cyc_name}: no track data")
            continue

        cyc_dir = get_cyclone_output_dir(cyc_name)
        safe_cyc_name = make_safe_name(cyc_name).lower()

        fig, ax = plt.subplots(
            figsize=(10, 8),
            subplot_kw={'projection': ccrs.PlateCarree()}
        )

        track = cyc['track']
        lons = [t[0] for t in track]
        lats = [t[1] for t in track]
        winds = [t[3] for t in track]

        valid_wind_values = [w for w in winds if w is not None]
        peak_wind = max(valid_wind_values) if valid_wind_values else None
        peak_idx = next((i for i, w in enumerate(winds) if w == peak_wind), None)

        lon_pad = 2
        lat_pad = 2
        ax.set_extent(
            [min(lons) - lon_pad, max(lons) + lon_pad, min(lats) - lat_pad, max(lats) + lat_pad],
            crs=ccrs.PlateCarree()
        )

        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linestyle='--', linewidth=0.5)

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', linestyle='--')
        gl.top_labels = False
        gl.right_labels = False

        ax.plot(
            lons, lats, '-',
            color=colors[cyc_name],
            linewidth=2,
            transform=ccrs.PlateCarree(),
            label=f"{cyc_name} Track"
        )

        for i, (lon, lat, dt, wind, pres) in enumerate(track):
            size = (wind / 10) ** 1.5 * 5 if wind is not None else 30

            ax.scatter(
                lon, lat,
                s=size,
                c=colors[cyc_name],
                marker=markers[cyc_name],
                edgecolor='white',
                linewidth=0.5,
                transform=ccrs.PlateCarree(),
                zorder=5
            )

            is_key_point = (i == 0) or (i == len(track) - 1) or (peak_idx is not None and i == peak_idx)

            if is_key_point:
                date_label = str(dt).split()[0][5:]
                ax.annotate(
                    date_label,
                    (lon, lat),
                    xytext=(5, 5),
                    textcoords='offset points',
                    fontsize=7,
                    color=colors[cyc_name]
                )

        if peak_idx is not None:
            ax.scatter(
                lons[peak_idx], lats[peak_idx],
                s=200,
                c='yellow',
                marker='*',
                edgecolor=colors[cyc_name],
                linewidth=1.5,
                transform=ccrs.PlateCarree(),
                zorder=6
            )

        ax.legend(loc='upper left', fontsize=9)
        ax.set_title(f'Cyclone Track: {cyc_name} ({YEAR})', fontsize=13, fontweight='bold')

        plt.tight_layout()

        out_file = os.path.join(cyc_dir, f'{safe_cyc_name}_track.png')
        fig.savefig(
            out_file,
            dpi=150,
            bbox_inches='tight',
            facecolor='white'
        )

        saved_files.append((cyc_name, out_file))
        print(f"Saved: {cyc_name} --> {out_file}")

        # plt.show()
        plt.close(fig)

    return saved_files

In [ ]:
# saved_track_files = plot_individual_cyclone_tracks()

# print("\n" + "=" * 70)
# print("CYCLONE TRACK FILES CREATED")
# print("=" * 70)

# for i, (cyc_name, file_path) in enumerate(saved_track_files, start=1):
#     print(f"{i:02d}. {cyc_name}")
#     print(f"    {file_path}")

In [ ]:
import matplotlib.patheffects as pe
import cmocean
track_cmap = cmocean.cm.phase
def plot_all_cyclone_tracks_global(show_bbox=True, save_path=None):
    '''
    Plot all cyclone tracks together on a single global map.

    Each cyclone receives a unique colour from the cmocean.cm.phase cyclic
    colormap.  Circle markers = start; star markers = end.

    Parameters
    ----------
    show_bbox : bool, optional
        Draw a dashed bounding box around all track points.  Default True.
    save_path : str or None, optional
        Save path if provided.

    Returns
    -------
    matplotlib.figure.Figure or None
    '''
    
    fig = plt.figure(figsize=(14.5, 7.5))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_global()
    ax.add_feature(cfeature.OCEAN, facecolor='#a8c7e6', zorder=0)
    ax.add_feature(cfeature.LAND, facecolor='#d9d3c1', zorder=1)
    ax.coastlines(linewidth=0.6, zorder=2)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':', zorder=2)
    gl = ax.gridlines(draw_labels=True, linewidth=0.2, color='gray', alpha=0.35, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 7}
    gl.ylabel_style = {'size': 7}

    # Build list of tracks and bounds
    all_lons = []
    all_lats = []
    tracks = []
    for cyc_name, cyc in cyclones.items():
        track = cyc.get('track', [])
        if not track:
            continue
        lons = [t[0] for t in track]
        lats = [t[1] for t in track]
        tracks.append((cyc_name, lons, lats))
        all_lons.extend(lons)
        all_lats.extend(lats)

    if not tracks:
        print("No cyclone tracks found.")
        return None

    # Color per cyclone
    colors = [track_cmap(i / max(1, len(tracks) - 1)) for i in range(len(tracks))]

    for (cyc_name, lons, lats), color in zip(tracks, colors):
        ax.plot(
            lons, lats,
            color=color, linewidth=1.4, alpha=0.95,
            transform=ccrs.Geodetic(), label=cyc_name,
            path_effects=[pe.withStroke(linewidth=2.4, foreground='white', alpha=0.6)]
        )
        ax.plot(lons[0], lats[0], 'o', color=color, markersize=3.0, transform=ccrs.PlateCarree())
        ax.plot(lons[-1], lats[-1], '*', color=color, markersize=4.5, transform=ccrs.PlateCarree())

    if show_bbox and all_lons and all_lats:
        pad = 2.0
        lon_min = min(all_lons) - pad
        lon_max = max(all_lons) + pad
        lat_min = min(all_lats) - pad
        lat_max = max(all_lats) + pad
        rect = plt.Rectangle(
            (lon_min, lat_min),
            lon_max - lon_min,
            lat_max - lat_min,
            fill=False, edgecolor='#333333', linewidth=1.2, linestyle='--',
            transform=ccrs.PlateCarree(), zorder=3
        )
        ax.add_patch(rect)

    ax.set_title(f'Cyclone Tracks in Year {YEAR}', fontsize=14, fontweight='bold')
    ax.legend(loc='lower left', fontsize=6, ncol=3, framealpha=0.8)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"Saved -> {save_path}")

    return fig

# Example:

# fig = plot_all_cyclone_tracks_global(show_bbox=False, save_path=os.path.join(OUTPUT_DIR, "all_cyclone_tracks_global.png"))
# plt.show()

## 1.6 First-Look: Side-by-Side SST and MSLA RAW Plots

## 1.6.1 Generate First-Look Plots for Cyclone

In [ ]:
def _get_phase_date_ranges(cyc):
    '''
    Extract pre / during / post phase date ranges for one cyclone.

    Prefers explicit analysis_dates in the cyclone record; falls back to
    deriving the three phases from individual date fields.

    Parameters
    ----------
    cyc : dict
        Normalised cyclone record as returned by load_cyclones.

    Returns
    -------
    dict or None
        {pre: (start, end), during: (start, end), post: (start, end)} where
        each value is a 2-tuple of ISO date strings, or None if insufficient
        date information is available.
    '''
    if cyc.get("analysis_dates"):
        return cyc["analysis_dates"]

    dates = cyc.get("dates", {})
    formation       = dates.get("formation")
    dissipation     = dates.get("dissipation")
    landfall        = dates.get("landfall")
    start_analysis  = dates.get("start_analysis_date")
    end_analysis    = dates.get("end_analysis_date")

    if not formation or not dissipation or not start_analysis or not end_analysis:
        return None

    formation_dt      = datetime.strptime(formation,      "%Y-%m-%d")
    dissipation_dt    = datetime.strptime(dissipation,    "%Y-%m-%d")
    landfall_dt       = datetime.strptime(landfall,       "%Y-%m-%d") if landfall else dissipation_dt
    start_analysis_dt = datetime.strptime(start_analysis, "%Y-%m-%d")
    end_analysis_dt   = datetime.strptime(end_analysis,   "%Y-%m-%d")

    return {
        "pre": (
            start_analysis_dt.strftime("%Y-%m-%d"),
            (formation_dt - timedelta(days=1)).strftime("%Y-%m-%d"),
        ),
        "during": (
            formation_dt.strftime("%Y-%m-%d"),
            dissipation_dt.strftime("%Y-%m-%d"),
        ),
        "post": (
            (dissipation_dt + timedelta(days=1)).strftime("%Y-%m-%d"),
            end_analysis_dt.strftime("%Y-%m-%d"),
        ),
    }


def build_phase_date_ranges_for_all(cyclones_dict):
    '''
    Build phase date ranges for all cyclones, skipping those with missing dates.

    Parameters
    ----------
    cyclones_dict : dict
        {cyclone_name: normalised_record} as returned by load_cyclones.

    Returns
    -------
    dict
        {cyclone_name: phase_date_ranges_dict}
    '''
    ranges = {}
    for cyclone_name, cyc in cyclones_dict.items():
        phase_ranges = _get_phase_date_ranges(cyc)
        if phase_ranges is None:
            continue
        ranges[cyclone_name] = phase_ranges
    return ranges


# all_phase_date_ranges = build_phase_date_ranges_for_all(cyclones)
# print(f"Built phase date ranges for {len(all_phase_date_ranges)} cyclones.")


In [ ]:
def _make_square_bbox(bbox, track=None, pad_deg=0.6):
    '''
    Expand a bounding box to include the track and return a square domain.

    Parameters
    ----------
    bbox : dict
        Initial bounding box: lon_min, lon_max, lat_min, lat_max.
    track : list of tuple or None, optional
        Track points as (lon, lat, datetime_str, wind, pressure).
    pad_deg : float, optional
        Degrees of padding on every side.  Default 0.6.

    Returns
    -------
    dict
        Square bounding box with the same keys as input bbox.
    '''
    lon_min, lon_max = bbox['lon_min'], bbox['lon_max']
    lat_min, lat_max = bbox['lat_min'], bbox['lat_max']
    if track:
        track_lons = [t[0] for t in track]; track_lats = [t[1] for t in track]
        lon_min = min(lon_min, min(track_lons)); lon_max = max(lon_max, max(track_lons))
        lat_min = min(lat_min, min(track_lats)); lat_max = max(lat_max, max(track_lats))
    lon_min -= pad_deg; lon_max += pad_deg
    lat_min -= pad_deg; lat_max += pad_deg
    lon_span = lon_max - lon_min; lat_span = lat_max - lat_min
    span     = max(lon_span, lat_span, 0.01)
    lon_mid  = (lon_min + lon_max) / 2.0; lat_mid = (lat_min + lat_max) / 2.0
    half     = span / 2.0
    return {'lon_min': lon_mid - half, 'lon_max': lon_mid + half,
            'lat_min': lat_mid - half, 'lat_max': lat_mid + half}


def _add_phase_date_label(ax, phase_upper, date_txt, phase_color):
    '''
    Annotate a subplot with a coloured phase badge and date text.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
    phase_upper : str
        Phase label in uppercase (PRE, DURING, or POST).
    date_txt : str
        Short date string (e.g. May 26).
    phase_color : str
        Hex colour for the badge background.
    '''
    ax.text(0.5, 1.16, phase_upper, transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8.2, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.25', facecolor=phase_color,
                      edgecolor='none', alpha=0.95),
            path_effects=[pe.withStroke(linewidth=1.4, foreground='black')],
            clip_on=False)
    ax.text(0.5, 1.05, date_txt, transform=ax.transAxes,
            ha='center', va='bottom', fontsize=7.6, fontweight='bold',
            color='black', clip_on=False)


def _draw_panel(ax, data, bbox, cmap, vmin, vmax, track_data=None,
                date=None, show_title=''):
    '''
    Draw a single map panel with optional data field and track overlay.

    Parameters
    ----------
    ax : cartopy.mpl.geoaxes.GeoAxes
    data : xr.DataArray or None
    bbox : dict
    cmap : str or Colormap
    vmin, vmax : float
    track_data : list of tuple or None, optional
    date : datetime.datetime or None, optional
    show_title : str, optional

    Returns
    -------
    matplotlib.collections.QuadMesh or None
    '''
    ax.set_extent([bbox['lon_min'], bbox['lon_max'], bbox['lat_min'], bbox['lat_max']],
                  crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor='#d9d3c1', zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=2)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.3, linestyle=':', zorder=2)
    ax.gridlines(draw_labels=False, linewidth=0.2, color='gray', alpha=0.35, linestyle='--')
    im = None
    if data is not None:
        lon_name = 'lon' if 'lon' in data.coords else 'longitude'
        lat_name = 'lat' if 'lat' in data.coords else 'latitude'
        im = ax.pcolormesh(data[lon_name].values, data[lat_name].values, data.values,
                           cmap=cmap, vmin=vmin, vmax=vmax,
                           transform=ccrs.PlateCarree(), shading='auto', zorder=0)
    else:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center', fontsize=8, color='gray')
    if track_data and date is not None:
        all_lons = [t[0] for t in track_data]; all_lats = [t[1] for t in track_data]
        all_dts  = [datetime.strptime(t[2], '%Y-%m-%d %H:%M') for t in track_data]
        ax.plot(all_lons, all_lats, '--', color='white', lw=0.6, alpha=0.4,
                transform=ccrs.PlateCarree(), zorder=3)
        past_lons = [lo for lo, dt in zip(all_lons, all_dts) if dt.date() <= date.date()]
        past_lats = [la for la, dt in zip(all_lats, all_dts) if dt.date() <= date.date()]
        if past_lons:
            ax.plot(past_lons, past_lats, '-', color='yellow', lw=1.0,
                    transform=ccrs.PlateCarree(), zorder=4)
            ax.plot(past_lons[-1], past_lats[-1], '*', color='red', ms=5,
                    transform=ccrs.PlateCarree(), zorder=5)
    if show_title:
        ax.set_title(show_title, fontsize=8)
    return im


def generate_cyclone_panel_figure(cyclone_name, phase_date_ranges,
                                  save_plots=True, output_dir=None):
    '''
    Generate a side-by-side SST and MSLA panel figure for one cyclone.

    Produces a grid with day-columns x 2 row-groups (SST top / MSLA bottom),
    colour-coded by phase (pre / during / post).

    Parameters
    ----------
    cyclone_name : str
        Key into the module-level cyclones dict (uppercase).
    phase_date_ranges : dict
        {pre: (start, end), during: (start, end), post: (start, end)}.
    save_plots : bool, optional
        Save the figure.  Default True.
    output_dir : str or None, optional
        Override save directory.

    Returns
    -------
    matplotlib.figure.Figure
    '''
    cyc   = cyclones[cyclone_name]
    track = cyc.get('track', [])
    bbox  = _make_square_bbox(cyc['bbox'], track=track)
    sst_vmin = globals().get('SST_VMIN', 28.0); sst_vmax = globals().get('SST_VMAX', 34.0)
    sla_vmin = globals().get('SLA_VMIN', -0.3); sla_vmax = globals().get('SLA_VMAX',  0.3)
    dates_info     = cyc.get('dates', {})
    start_analysis = dates_info.get('start_analysis_date')
    end_analysis   = dates_info.get('end_analysis_date')
    formation      = dates_info.get('formation')
    dissipation    = dates_info.get('dissipation')
    ordered_days   = []
    if start_analysis and end_analysis and formation and dissipation:
        start_dt       = datetime.strptime(start_analysis, '%Y-%m-%d')
        end_dt         = datetime.strptime(end_analysis,   '%Y-%m-%d')
        formation_dt   = datetime.strptime(formation,      '%Y-%m-%d')
        dissipation_dt = datetime.strptime(dissipation,    '%Y-%m-%d')
        cur = start_dt
        while cur <= end_dt:
            phase = 'pre' if cur < formation_dt else ('during' if cur <= dissipation_dt else 'post')
            ordered_days.append((cur, phase)); cur += timedelta(days=1)
    else:
        for phase, (s_str, e_str) in phase_date_ranges.items():
            cur = datetime.strptime(s_str, '%Y-%m-%d'); e_dt = datetime.strptime(e_str, '%Y-%m-%d')
            while cur <= e_dt:
                ordered_days.append((cur, phase)); cur += timedelta(days=1)
    n_days      = len(ordered_days)
    target_days = 12 if n_days <= 12 else 20
    while len(ordered_days) < target_days: ordered_days.append((None, None))
    ordered_days = ordered_days[:target_days]
    phase_colors = {'pre': '#4a90d9', 'during': '#e05c2a', 'post': '#3daa5e'}
    cmap_msla = 'RdBu_r'
    try:
        import cmocean; cmap_msla = cmocean.cm.balance
    except ImportError: pass
    rows = 3 if target_days == 12 else 5
    fig  = plt.figure(figsize=(18, 2.25 * rows + 2.2))
    gs   = gridspec.GridSpec(rows, 8, hspace=0.42, wspace=0.08,
                             left=0.06, right=0.94, top=0.88, bottom=0.17)
    sst_im_ref = msla_im_ref = None
    for r in range(rows):
        for d in range(4):
            date, phase = ordered_days[r * 4 + d]
            phase_upper = phase.upper() if phase else ''
            phase_color = phase_colors.get(phase, '#888888')
            date_txt    = date.strftime('%b %d') if date else ''
            sst_ax  = fig.add_subplot(gs[r, d],     projection=ccrs.PlateCarree())
            msla_ax = fig.add_subplot(gs[r, d + 4], projection=ccrs.PlateCarree())
            sst  = load_sst_for_date(date, bbox)  if date else None
            msla = load_msla_for_date(date, bbox) if date else None
            im_s = _draw_panel(sst_ax,  sst,  bbox, 'RdYlBu_r', sst_vmin, sst_vmax, track, date)
            im_m = _draw_panel(msla_ax, msla, bbox, cmap_msla,   sla_vmin, sla_vmax, track, date)
            if im_s: sst_im_ref  = im_s
            if im_m: msla_im_ref = im_m
            if date:
                _add_phase_date_label(sst_ax,  phase_upper, date_txt, phase_color)
                _add_phase_date_label(msla_ax, phase_upper, date_txt, phase_color)
    for cax_bbox, ref, label in [
        ([0.08, 0.07, 0.38, 0.025], sst_im_ref,  'SST (C)'),
        ([0.54, 0.07, 0.38, 0.025], msla_im_ref, 'SLA (m)'),
    ]:
        cax = fig.add_axes(cax_bbox)
        if ref is not None:
            cb = fig.colorbar(ref, cax=cax, orientation='horizontal', extend='both')
            cb.set_label(label, fontsize=8.5); cb.ax.tick_params(labelsize=7)
    title_bits = [cyc['name'], 'SST & MSLA Time Series']
    if start_analysis and end_analysis:
        title_bits.append(f"Analysis: {start_analysis} to {end_analysis}")
    fig.suptitle(' | '.join(title_bits), fontsize=12.5, fontweight='bold', y=0.955)
    if save_plots:
        out   = output_dir or get_cyclone_output_dir(cyclone_name)
        fpath = os.path.join(out, f'{make_safe_name(cyclone_name)}_SST_MSLA_panel.png')
        fig.savefig(fpath, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"Saved -> {fpath}")
    return fig


In [ ]:
# panel_figures = {}

# for cyclone_name, cyc in cyclones.items():
#     phase_ranges = _get_phase_date_ranges(cyc)
#     if phase_ranges is None:
#         print(f"Skipping {cyclone_name}: no usable date ranges")
#         continue

#     fig = generate_cyclone_panel_figure(
#         cyclone_name,
#         phase_ranges,
#         save_plots=False
#     )
#     panel_figures[cyclone_name] = fig

# # plt.show()

---
# PHASE 2: 2D Gaussian Filter for Scale Separation

## 2.1 Scale Decomposition Main Function

In [ ]:
def scale_decomposition(field_2d, grid_resolution_deg,
                        filter_scales=None, ref_lat=19.0,
                        mode='reflect', truncate=4.0,
                        verbose=True):
    '''
    Decompose a 2-D oceanographic field into sub-mesoscale, mesoscale,
    and large-scale components using nested Gaussian low-pass filtering.

    Decomposition identity (exact reconstruction guaranteed):
        submeso + meso + large_scale == raw_field

    Band definitions
    ----------------
    large_scale = Gaussian low-pass at large scale
    meso        = low-pass at meso scale minus large_scale (band-pass)
    submeso     = raw field minus low-pass at meso scale   (high-pass)

    NaN handling: NaN pixels filled with domain mean before filtering and
    re-masked in output.  Anisotropic sigma accounts for 1 deg lon != 1 deg
    lat at non-zero latitudes.

    Parameters
    ----------
    field_2d : np.ndarray or xr.DataArray  shape (lat, lon)
    grid_resolution_deg : float
        Grid spacing in decimal degrees (e.g. 0.05 for downsampled MUR SST).
    filter_scales : dict or None, optional
        {submeso, meso, large} cutoff scales in degrees.
        Defaults to FILTER_SCALES from the Configuration cell.
    ref_lat : float, optional
        Reference latitude for anisotropic sigma.  Default 19.0.
    mode : str, optional
        Boundary mode for gaussian_filter.  Default reflect.
    truncate : float, optional
        Kernel half-width in sigma units.  Default 4.0.
    verbose : bool, optional
        Print sigma values and residual.  Default True.

    Returns
    -------
    dict
        Keys: raw, large_scale, meso, submeso, sigma_sub, sigma_meso,
        sigma_large.  Spatial arrays are 2-D NumPy ndarrays; sigma values
        are (sigma_lat, sigma_lon) tuples.
    '''
    if hasattr(field_2d, 'values'):
        data = field_2d.values.astype(float).copy()
    else:
        data = np.asarray(field_2d, dtype=float).copy()

    if filter_scales is None:
        filter_scales = FILTER_SCALES

    mask        = np.isnan(data)
    fill_val    = np.nanmean(data)
    data_filled = np.where(mask, fill_val, data)

    dlat_km = 111.0 * grid_resolution_deg
    dlon_km = 111.0 * np.cos(np.deg2rad(ref_lat)) * grid_resolution_deg

    def _sigma_pair(scale_deg):
        scale_km = 111.0 * scale_deg
        return (scale_km / dlat_km, scale_km / dlon_km)

    sigma_sub   = _sigma_pair(filter_scales['submeso'])
    sigma_meso  = _sigma_pair(filter_scales['meso'])
    sigma_large = _sigma_pair(filter_scales['large'])

    if verbose:
        print(f"  sigma_sub   lat:{sigma_sub[0]:.2f}  lon:{sigma_sub[1]:.2f}  grid pts")
        print(f"  sigma_meso  lat:{sigma_meso[0]:.2f}  lon:{sigma_meso[1]:.2f}  grid pts")
        print(f"  sigma_large lat:{sigma_large[0]:.2f}  lon:{sigma_large[1]:.2f}  grid pts")

    sub_lp   = gaussian_filter(data_filled, sigma=sigma_sub,   mode=mode, truncate=truncate)
    meso_lp  = gaussian_filter(data_filled, sigma=sigma_meso,  mode=mode, truncate=truncate)
    large_lp = gaussian_filter(data_filled, sigma=sigma_large, mode=mode, truncate=truncate)

    submeso = data_filled - meso_lp
    meso    = meso_lp  - large_lp
    large   = large_lp

    if verbose:
        residual = np.nanmean(np.abs(submeso + meso + large - data_filled))
        print(f"  Reconstruction residual: {residual:.2e}  {'OK' if residual < 1e-8 else 'CHECK'}")

    return {
        'raw'        : np.where(mask, np.nan, data),
        'large_scale': np.where(mask, np.nan, large),
        'meso'       : np.where(mask, np.nan, meso),
        'submeso'    : np.where(mask, np.nan, submeso),
        'sigma_sub'  : sigma_sub,
        'sigma_meso' : sigma_meso,
        'sigma_large': sigma_large,
    }


def decompose_sst_field(sst_data, ref_lat=19.0):
    '''
    Apply scale decomposition to a MUR-JPL SST DataArray (0.05 deg grid).

    Parameters
    ----------
    sst_data : xr.DataArray or np.ndarray  shape (lat, lon)
        SST in degrees Celsius.
    ref_lat : float, optional
        Fallback reference latitude.  Default 19.0.

    Returns
    -------
    dict
        Scale decomposition result using FILTER_SCALES.
    '''
    grid_res = 0.05
    if hasattr(sst_data, 'lat'):
        ref_lat = float(sst_data.lat.mean())
    print(f"  SST  grid_res={grid_res} deg  ref_lat={ref_lat:.1f} N")
    return scale_decomposition(sst_data, grid_resolution_deg=grid_res,
                               filter_scales=FILTER_SCALES, ref_lat=ref_lat)


def decompose_msla_field(msla_data, ref_lat=19.0):
    '''
    Apply scale decomposition to a CMEMS DUACS MSLA DataArray (0.125 deg grid).

    Parameters
    ----------
    msla_data : xr.DataArray or np.ndarray  shape (latitude, longitude)
        SLA in metres.
    ref_lat : float, optional
        Fallback reference latitude.  Default 19.0.

    Returns
    -------
    dict
        Scale decomposition result using FILTER_SCALES_MSLA.
    '''
    grid_res = 0.125
    if hasattr(msla_data, 'latitude'):
        ref_lat = float(msla_data.latitude.mean())
    elif hasattr(msla_data, 'lat'):
        ref_lat = float(msla_data.lat.mean())
    print(f"  MSLA grid_res={grid_res} deg  ref_lat={ref_lat:.1f} N")
    return scale_decomposition(msla_data, grid_resolution_deg=grid_res,
                               filter_scales=FILTER_SCALES_MSLA, ref_lat=ref_lat)


print("Scale decomposition functions defined.")


---
# PHASE 3: Scale-wise Visual Comparison

## 3.1 Multi-Scale Visualization Function

## 3.1 Comprehensive Scale Evolution Matrix Plot

Generate **two separate 4 —12 or 4- 20 matrix figures** per cyclone:
- **SST** scale evolution (Raw, Large, Meso, Sub-meso)
- **MSLA** scale evolution (Raw, Large, Meso, Sub-meso) 

Columns span pre,  during,  post cyclone phases for respective dates.

In [ ]:
def _get_analysis_window_ranges(cyc):
    '''
    Extract all named analysis window date ranges from a cyclone record.

    Parameters
    ----------
    cyc : dict
        Normalised cyclone record as returned by load_cyclones.

    Returns
    -------
    list of tuple
        [(window_label, start_str, end_str), ...].
        Empty list when no usable date information can be found.
    '''
    dates  = cyc.get('dates', {})
    ranges = []

    def _add_range(label, start_str, end_str):
        if start_str and end_str:
            ranges.append((label, start_str, end_str))

    _add_range('ANALYSIS_12D', dates.get('start_analysis_date_12'), dates.get('end_analysis_date_12'))
    _add_range('ANALYSIS_20D', dates.get('start_analysis_date_20'), dates.get('end_analysis_date_20'))
    _add_range('ANALYSIS',     dates.get('start_analysis_date'),    dates.get('end_analysis_date'))

    for key in ['analysis_window_12', 'analysis_window_20', 'analysis_window']:
        win = dates.get(key) if isinstance(dates.get(key), dict) else None
        if win:
            _add_range(key.upper(), win.get('start'), win.get('end'))

    if not ranges:
        analysis_dates = cyc.get('analysis_dates') or {}
        all_dates = []
        for phase in ['pre', 'during', 'post']:
            if phase in analysis_dates:
                s_str, e_str = analysis_dates[phase]
                if s_str: all_dates.append(s_str)
                if e_str: all_dates.append(e_str)
        if all_dates:
            _add_range('ANALYSIS', min(all_dates), max(all_dates))

    return ranges


def _get_analysis_date_list_from_range(start_str, end_str):
    '''
    Generate a list of daily datetime objects for a date range.

    Parameters
    ----------
    start_str : str
        Range start in YYYY-MM-DD format (inclusive).
    end_str : str
        Range end in YYYY-MM-DD format (inclusive).

    Returns
    -------
    list of datetime.datetime
    '''
    return [d.to_pydatetime() for d in pd.date_range(start_str, end_str, freq='D')]


def _resolve_cyclone_output_dir(cyclone_name, output_dir=None):
    '''
    Determine the output directory for a cyclone.

    Checks CYCLONE_OUTPUT_DIRS cache first, then falls back to OUTPUT_DIR.

    Parameters
    ----------
    cyclone_name : str
        Uppercase cyclone name.
    output_dir : str or None, optional
        Override base directory.

    Returns
    -------
    str
        Resolved output directory path.
    '''
    if 'CYCLONE_OUTPUT_DIRS' in globals() and cyclone_name in CYCLONE_OUTPUT_DIRS:
        return CYCLONE_OUTPUT_DIRS[cyclone_name]
    base      = output_dir or OUTPUT_DIR
    candidate = os.path.join(base, cyclone_name)
    if os.path.isdir(candidate):
        return candidate
    return base


def _draw_evolution_panel(ax, field, lons, lats, bbox,
                          cmap, vmin, vmax, diverge,
                          track_data=None, date=None,
                          add_left_labels=False, add_bottom_labels=False):
    '''
    Render one cell of a multi-column scale-evolution matrix figure.

    Parameters
    ----------
    ax : cartopy.mpl.geoaxes.GeoAxes
    field : np.ndarray or None
        2-D decomposed field array.  None renders a No data label.
    lons, lats : np.ndarray or None
    bbox : dict
    cmap : str or Colormap
    vmin, vmax : float
    diverge : bool
        When True, applies TwoSlopeNorm centred on zero.
    track_data : list of tuple or None, optional
    date : datetime.datetime or None, optional
    add_left_labels : bool, optional
    add_bottom_labels : bool, optional

    Returns
    -------
    matplotlib.collections.QuadMesh or None
    '''
    ax.set_extent([bbox['lon_min'], bbox['lon_max'], bbox['lat_min'], bbox['lat_max']],
                  crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor='#d4c9a8', zorder=3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=4)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.3, linestyle=':', zorder=4)
    gl = ax.gridlines(draw_labels=False, linewidth=0.2, color='gray', alpha=0.35, linestyle='--')
    if add_left_labels:
        gl.left_labels  = True; gl.ylabel_style = {'size': 6}
    if add_bottom_labels:
        gl.bottom_labels = True; gl.xlabel_style = {'size': 6}
    if field is not None and lons is not None and lats is not None:
        norm = (TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
                if diverge else Normalize(vmin=vmin, vmax=vmax))
        im = ax.pcolormesh(lons, lats, field, cmap=cmap, norm=norm,
                           transform=ccrs.PlateCarree(), shading='auto', zorder=1)
    else:
        im = None
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center', fontsize=8, color='gray')
    if track_data and date is not None:
        all_lons = [e[0] for e in track_data]; all_lats = [e[1] for e in track_data]
        all_dts  = [datetime.strptime(e[2], '%Y-%m-%d %H:%M') for e in track_data]
        ax.plot(all_lons, all_lats, '--', color='white', lw=0.8, alpha=0.4,
                transform=ccrs.PlateCarree(), zorder=5)
        past_lons = [lo for lo, dt in zip(all_lons, all_dts) if dt.date() <= date.date()]
        past_lats = [la for la, dt in zip(all_lats, all_dts) if dt.date() <= date.date()]
        if past_lons:
            ax.plot(past_lons, past_lats, '-o', color='yellow', lw=1.2, ms=2.5,
                    transform=ccrs.PlateCarree(), zorder=6)
            ax.plot(past_lons[-1], past_lats[-1], '*', color='red', ms=7.0,
                    transform=ccrs.PlateCarree(), zorder=7)
    return im


def _plot_scale_evolution_matrix_one(cyclone_name, variable='SST',
                                     save=True, output_dir=None):
    '''
    Generate the multi-scale spatial evolution matrix for one cyclone.

    Produces a figure: n_rows (decomposition scales) x n_cols (calendar days).
    Rows = raw, large-scale, meso, sub-meso; columns = analysis window days.

    Parameters
    ----------
    cyclone_name : str
        Uppercase cyclone name.
    variable : str, optional
        SST or MSLA.  Default SST.
    save : bool, optional
        Save and close the figure.  Default True.
    output_dir : str or None, optional
        Override output directory.

    Returns
    -------
    dict or None
        {window_label: saved_filepath_or_figure} for each analysis window.
    '''
    var   = variable.upper()
    cyc   = cyclones[cyclone_name]
    track = cyc.get('track', [])
    bbox  = _make_square_bbox(cyc['bbox'], track=track)

    windows = _get_analysis_window_ranges(cyc)
    if not windows:
        print(f"Skipping {cyclone_name}: no analysis window dates in JSON")
        return None

    dates_info     = cyc.get('dates', {})
    formation      = dates_info.get('formation')
    dissipation    = dates_info.get('dissipation')
    formation_dt   = datetime.strptime(formation,   '%Y-%m-%d') if formation   else None
    dissipation_dt = datetime.strptime(dissipation, '%Y-%m-%d') if dissipation else None

    phase_colors = {'pre': '#4a90d9', 'during': '#e05c2a', 'post': '#3daa5e'}
    figures      = {}
    cyclone_out_dir = _resolve_cyclone_output_dir(cyclone_name, output_dir)

    for window_label, start_str, end_str in windows:
        date_list = _get_analysis_date_list_from_range(start_str, end_str)
        n_cols    = len(date_list)
        n_rows    = len(SCALE_ROWS)

        print(f"\n{'=' * 65}")
        print(f"  {cyclone_name}  |  {var}  |  {n_cols} cols x {n_rows} rows")
        print(f"  Window: {window_label}  ({start_str} -> {end_str})")
        print(f"{'=' * 65}")

        panel_w = 3.0
        fig_w   = max(26, n_cols * panel_w + 2.0)
        fig_h   = n_rows * 4.6 + 1.6
        fig     = plt.figure(figsize=(fig_w, fig_h))

        gs = gridspec.GridSpec(
            n_rows + 1, n_cols,
            height_ratios=[0.055] + [1] * n_rows,
            hspace=0.22, wspace=0.03,
            left=0.05, right=0.90, top=0.91, bottom=0.05
        )

        hax = fig.add_subplot(gs[0, :])
        hax.set_facecolor(ANALYSIS_COLOR); hax.set_axis_off()
        hax.text(0.5, 0.5, window_label, transform=hax.transAxes,
                 ha='center', va='center', fontsize=12, fontweight='bold',
                 color=ANALYSIS_TEXT,
                 path_effects=[pe.withStroke(linewidth=2, foreground='black')])

        cbar_row_h = 0.82 / n_rows
        cbar_axes  = [
            fig.add_axes([0.92, 0.05 + (n_rows - 1 - ri) * cbar_row_h + cbar_row_h * 0.12,
                          0.012, cbar_row_h * 0.70])
            for ri in range(n_rows)
        ]
        row_ims = [None] * n_rows

        for ci, date in enumerate(date_list):
            print(f"  col {ci+1:2d}/{n_cols}  {date.strftime('%Y-%m-%d')} ...", end=' ', flush=True)
            if formation_dt and dissipation_dt:
                phase = ('pre' if date < formation_dt else
                         'during' if date <= dissipation_dt else 'post')
            else:
                phase = 'during'

            if var == 'SST':
                raw_data = load_sst_for_date(date, bbox)
                decomp   = decompose_sst_field(raw_data) if raw_data is not None else None
                lons = raw_data.lon.values      if raw_data is not None else None
                lats = raw_data.lat.values      if raw_data is not None else None
            else:
                raw_data = load_msla_for_date(date, bbox)
                decomp   = decompose_msla_field(raw_data) if raw_data is not None else None
                lons = raw_data.longitude.values if raw_data is not None else None
                lats = raw_data.latitude.values  if raw_data is not None else None

            if decomp is not None and 'large_scale' in decomp and var == 'SST':
                decomp['large_scale'] = decomp['large_scale'] - np.nanmean(decomp['large_scale'])

            for ri, row_cfg in enumerate(SCALE_ROWS):
                ax  = fig.add_subplot(gs[ri + 1, ci], projection=ccrs.PlateCarree())
                for spine in ax.spines.values():
                    spine.set_edgecolor(ANALYSIS_COLOR); spine.set_linewidth(1.2)
                cfg   = row_cfg[var.lower()]
                field = decomp[row_cfg['key']] if decomp is not None else None
                im = _draw_evolution_panel(
                    ax, field, lons, lats, bbox,
                    cmap=cfg['cmap'], vmin=cfg['vmin'], vmax=cfg['vmax'],
                    diverge=cfg['diverge'], track_data=track, date=date,
                    add_left_labels=(ci == 0), add_bottom_labels=(ri == n_rows - 1))
                if im is not None:
                    row_ims[ri] = im
                if ri == 0:
                    _add_phase_date_label(ax, phase.upper(), date.strftime('%b %d'),
                                          phase_colors.get(phase, '#888888'))
                if ci == 0:
                    ax.text(-0.24, 0.5, row_cfg['label'],
                            transform=ax.transAxes, va='center', ha='center',
                            rotation=90, fontsize=8.2, fontweight='bold', color='#333333')

            print('done')
            if raw_data is not None: del raw_data
            if decomp is not None:   del decomp
            gc.collect()

        for ri, row_cfg in enumerate(SCALE_ROWS):
            cfg = row_cfg[var.lower()]
            if row_ims[ri] is not None:
                cb = fig.colorbar(row_ims[ri], cax=cbar_axes[ri],
                                  orientation='vertical', extend='both')
                cb.set_label(cfg['unit'], fontsize=8.0); cb.ax.tick_params(labelsize=7.0)

        fig.suptitle(
            f"{cyc['name']}  --  {var} Scale Decomposition Evolution\n{window_label} {start_str} -> {end_str}",
            fontsize=14, fontweight='bold', y=0.98)

        if save:
            out   = cyclone_out_dir or OUTPUT_DIR
            fname = f'{cyclone_name}_{var}_{window_label}_4x{n_cols}.png'
            fpath = os.path.join(out, fname)
            fig.savefig(fpath, dpi=150, bbox_inches='tight', facecolor='white')
            print(f"\n  Saved -> {fpath}")
            plt.close(fig); gc.collect()
            figures[window_label] = fpath
        else:
            figures[window_label] = fig

    return figures


def plot_scale_evolution_matrix(cyclone_name='ALL', variable='SST',
                                save=True, output_dir=None):
    '''
    Top-level runner for the scale-evolution matrix.

    Parameters
    ----------
    cyclone_name : str or list or None, optional
        ALL (or None) for every cyclone, a single name, or a list.
        Default ALL.
    variable : str, optional
        SST or MSLA.  Default SST.
    save : bool, optional
        Save figures to disk.  Default True.
    output_dir : str or None, optional
        Override output directory.
    '''
    if cyclone_name in ('ALL', None):
        results = {}
        for name in [c for c, info in cyclones.items() if info.get('dates')]:
            results[name] = _plot_scale_evolution_matrix_one(
                name, variable=variable, save=save, output_dir=output_dir)
            plt.close('all'); gc.collect()
        return results
    if isinstance(cyclone_name, (list, tuple, set)):
        results = {}
        for name in cyclone_name:
            if name not in cyclones:
                print(f"Skipping {name}: missing cyclones entry"); continue
            results[name] = _plot_scale_evolution_matrix_one(
                name, variable=variable, save=save, output_dir=output_dir)
            plt.close('all'); gc.collect()
        return results
    res = _plot_scale_evolution_matrix_one(
        cyclone_name, variable=variable, save=save, output_dir=output_dir)
    plt.close('all'); gc.collect()
    return res


# =============================================================================
# RUN  (uncomment to execute)
# =============================================================================
# for var in ['SST', 'MSLA']:
#     figs_by_cyclone = plot_scale_evolution_matrix('ALL', variable=var, save=True)
#     plt.close('all'); gc.collect()

print("\nAll Scale Evolution Matrix functions defined.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_gradient_magnitude(field2d, res_deg, ref_lat):
    """
    Computes the spatial gradient magnitude of a 2D field in physical units (per km).
    
    Args:
        field2d (np.ndarray): The 2D spatial field (e.g., large_scale, meso, or submeso).
        res_deg (float): Spatial resolution of the grid in degrees.
        ref_lat (float): Reference latitude for longitude distance scaling.
        
    Returns:
        np.ndarray: Gradient magnitude |∇T| in physical units (e.g. °C/km or m/km).
    """
    dy, dx = np.gradient(field2d)
    
    # Convert from units/gridcell to units/km
    dy_km = dy / (111.0 * res_deg)
    dx_km = dx / (111.0 * np.cos(np.deg2rad(ref_lat)) * res_deg)
    
    # Compute |∇T| = sqrt( (∂T/∂x)² + (∂T/∂y)² )
    grad_magnitude = np.sqrt(dy_km**2 + dx_km**2)
    
    return grad_magnitude

def process_cyclone_evolution(analysis_dates, bbox, ref_lat):
    '''
    Compute daily SST and MSLA decomposition history including gradient fields.

    For every date in analysis_dates loads raw SST and MSLA, applies scale
    decomposition, and computes gradient magnitude for each scale band via
    compute_gradient_magnitude.

    Parameters
    ----------
    analysis_dates : list of datetime.datetime
    bbox : dict
        Bounding box: lon_min, lon_max, lat_min, lat_max.
    ref_lat : float
        Reference latitude in degrees for anisotropic dx/dy computation.

    Returns
    -------
    sst_history : dict
        Keys: raw, large_scale, meso, submeso, large_grad, meso_grad, submeso_grad.
        Each value is a list of 2-D arrays or None per date.
    msla_history : dict
        Same structure as sst_history for the MSLA variable.
    '''
    sst_history = {'raw': [], 'large_scale': [], 'meso': [], 'submeso': [], 
                   'large_grad': [], 'meso_grad': [], 'submeso_grad': []}
    
    msla_history = {'raw': [], 'large_scale': [], 'meso': [], 'submeso': [], 
                    'large_grad': [], 'meso_grad': [], 'submeso_grad': []}
    
    for d in analysis_dates:
        # -----------------------------
        # 1. SST Processing (0.05° res)
        # -----------------------------
        sst_raw = load_sst_for_date(d, bbox)
        if sst_raw is not None:
            dec_sst = decompose_sst_field(sst_raw)
            if dec_sst:
                # Calculate gradients for all scales (SST resolution = 0.05°)
                res_sst = 0.05
                large_grad = compute_gradient_magnitude(dec_sst['large_scale'], res_sst, ref_lat)
                meso_grad  = compute_gradient_magnitude(dec_sst['meso'], res_sst, ref_lat)
                sub_grad   = compute_gradient_magnitude(dec_sst['submeso'], res_sst, ref_lat)
                
                sst_history['raw'].append(sst_raw.values)
                sst_history['large_scale'].append(dec_sst['large_scale'])
                sst_history['meso'].append(dec_sst['meso'])
                sst_history['submeso'].append(dec_sst['submeso'])
                
                sst_history['large_grad'].append(large_grad)
                sst_history['meso_grad'].append(meso_grad)
                sst_history['submeso_grad'].append(sub_grad)
            else:
                for k in sst_history: sst_history[k].append(None)
        else:
            for k in sst_history: sst_history[k].append(None)

        # ------------------------------
        # 2. MSLA Processing (0.125° res)
        # ------------------------------
        msla_raw = load_msla_for_date(d, bbox)
        if msla_raw is not None:
            dec_msla = decompose_msla_field(msla_raw)
            if dec_msla:
                # Calculate gradients for all scales (MSLA resolution = 0.125°)
                res_msla = 0.125 
                large_grad = compute_gradient_magnitude(dec_msla['large_scale'], res_msla, ref_lat)
                meso_grad  = compute_gradient_magnitude(dec_msla['meso'], res_msla, ref_lat)
                sub_grad   = compute_gradient_magnitude(dec_msla['submeso'], res_msla, ref_lat)

                msla_history['raw'].append(msla_raw.values)
                msla_history['large_scale'].append(dec_msla['large_scale'])
                msla_history['meso'].append(dec_msla['meso'])
                msla_history['submeso'].append(dec_msla['submeso'])
                
                msla_history['large_grad'].append(large_grad)
                msla_history['meso_grad'].append(meso_grad)
                msla_history['submeso_grad'].append(sub_grad)
            else:
                for k in msla_history: msla_history[k].append(None)
        else:
            for k in msla_history: msla_history[k].append(None)
            
    return sst_history, msla_history


def plot_temporal_evolution(history_dict, scale_key, dates, lons, lats, title_prefix, cmap='magma'):
    """
    Plots the temporal evolution of a specific scale (e.g. 'submeso_grad').
    """
    num_days = len(dates)
    fig, axes = plt.subplots(1, num_days, figsize=(5 * num_days, 5), sharey=True)
    
    if num_days == 1:
        axes = [axes]
        
    for i, (date, ax) in enumerate(zip(dates, axes)):
        data = history_dict[scale_key][i]
        
        if data is not None:
            # Create filled contour/pcolormesh plot
            im = ax.pcolormesh(lons, lats, data, cmap=cmap, shading='auto')
            ax.set_title(f"{date.strftime('%Y-%m-%d')}")
            ax.set_xlabel("Longitude")
            if i == 0:
                ax.set_ylabel("Latitude")
        else:
            ax.set_title(f"{date.strftime('%Y-%m-%d')}\n(Missing Data)")
            ax.axis('off')
            
    fig.suptitle(f"Daily Spatial Evolution: {title_prefix} ({scale_key})", fontsize=16)
    
    # Add a global colorbar for the row
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) 
    fig.colorbar(im, cax=cbar_ax, label="Gradient Magnitude")
    
    plt.show()

# --- Example Usage ---
# plot_temporal_evolution(sst_history, 'submeso_grad', analysis_dates, sst_lons, sst_lats, "SST")
# plot_temporal_evolution(msla_history, 'meso_grad', analysis_dates, msla_lons, msla_lats, "MSLA")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import gc
from matplotlib.colors import Normalize


def _plot_scale_gradient_matrix_one(cyclone_name, variable='SST',
                                     save=True, output_dir=None):
    '''
    Generate the gradient-magnitude evolution matrix for one cyclone.

    Produces n_rows (raw + 3 gradient scales) x n_cols (days) subplots.
    Colour limits come from SCALE_GRAD_ROWS in the Configuration cell.

    Parameters
    ----------
    cyclone_name : str
    variable : str, optional
        SST or MSLA.  Default SST.
    save : bool, optional
        Save and close the figure.  Default True.
    output_dir : str or None, optional

    Returns
    -------
    dict
        {window_label: saved_filepath_or_figure}
    '''
    var = variable.upper()
    cyc = cyclones[cyclone_name]
    track = cyc.get('track', [])
    bbox = _make_square_bbox(cyc['bbox'], track=track)
    ref_lat = (bbox['lat_min'] + bbox['lat_max']) / 2.0

    windows = _get_analysis_window_ranges(cyc)
    if not windows:
        print(f"Skipping {cyclone_name}: no analysis window dates")
        return None
        
    dates_info = cyc.get('dates', {})
    formation_dt = pd.to_datetime(dates_info.get('formation')) if dates_info.get('formation') else None
    dissipation_dt = pd.to_datetime(dates_info.get('dissipation')) if dates_info.get('dissipation') else None

    phase_colors = {'pre': '#4a90d9', 'during': '#e05c2a', 'post': '#3daa5e'}
    figures = {}
    cyclone_out_dir = _resolve_cyclone_output_dir(cyclone_name, output_dir)

    for window_label, start_str, end_str in windows:
        date_list = _get_analysis_date_list_from_range(start_str, end_str)
        n_cols = len(date_list)
        n_rows = len(SCALE_GRAD_ROWS)

        panel_w = 3.0
        fig_w   = max(26, n_cols * panel_w + 2.0)
        fig_h   = n_rows * 4.6 + 1.6
        fig = plt.figure(figsize=(fig_w, fig_h))

        gs = gridspec.GridSpec(
            n_rows + 1, n_cols,
            height_ratios=[0.055] + [1] * n_rows,
            hspace=0.22, wspace=0.03, left=0.05, right=0.90, top=0.91, bottom=0.05
        )

        # Header 
        hax = fig.add_subplot(gs[0, :])
        hax.set_facecolor(ANALYSIS_COLOR)
        hax.set_axis_off()
        hax.text(0.5, 0.5, window_label, transform=hax.transAxes,
                 ha='center', va='center', fontsize=12, fontweight='bold', color=ANALYSIS_TEXT)

        # Shared row colorbars setup
        cbar_row_h = 0.82 / n_rows
        cbar_axes = []
        for ri in range(n_rows):
            y0 = 0.05 + (n_rows - 1 - ri) * cbar_row_h + cbar_row_h * 0.12
            cax = fig.add_axes([0.92, y0, 0.012, cbar_row_h * 0.70])
            cbar_axes.append(cax)

        row_ims = [None] * n_rows
        
        # Determine Resolution based on active variable
        res_deg = 0.05 if var == 'SST' else 0.125
        
        for ci, date in enumerate(date_list):
            if formation_dt and dissipation_dt:
                phase = 'pre' if date < formation_dt else ('during' if date <= dissipation_dt else 'post')
            else:
                phase = 'during'

            date_txt = date.strftime('%b %d')

            # Fetch Target Data
            raw_data = load_sst_for_date(date, bbox) if var == 'SST' else load_msla_for_date(date, bbox)
            decomp = None
            if raw_data is not None:
                lons = raw_data.lon.values if var == 'SST' else raw_data.longitude.values
                lats = raw_data.lat.values if var == 'SST' else raw_data.latitude.values
                decomp = decompose_sst_field(raw_data) if var == 'SST' else decompose_msla_field(raw_data)
                
            # Perform Target Computations if valid field exists
            plot_dict = {'raw': raw_data.values if raw_data is not None else None}
            if decomp is not None:
                plot_dict['large_grad']   = compute_gradient_magnitude(decomp['large_scale'], res_deg, ref_lat)
                plot_dict['meso_grad']    = compute_gradient_magnitude(decomp['meso'], res_deg, ref_lat)
                plot_dict['submeso_grad'] = compute_gradient_magnitude(decomp['submeso'], res_deg, ref_lat)

            # Draw Iteration Subplots
            for ri, row_cfg in enumerate(SCALE_GRAD_ROWS):
                ax = fig.add_subplot(gs[ri + 1, ci], projection=ccrs.PlateCarree())

                # Border stylings
                for spine in ax.spines.values():
                    spine.set_edgecolor(ANALYSIS_COLOR)
                    spine.set_linewidth(1.2)

                cfg = row_cfg[var.lower()]
                target_field = plot_dict.get(row_cfg['key'], None)

                im = _draw_evolution_panel(
                    ax, target_field, lons if raw_data is not None else None, lats if raw_data is not None else None,
                    bbox, cmap=cfg['cmap'], vmin=cfg['vmin'], vmax=cfg['vmax'], diverge=cfg['diverge'],
                    track_data=track, date=date, add_left_labels=(ci == 0), add_bottom_labels=(ri == n_rows - 1)
                )
                
                if im is not None:
                    row_ims[ri] = im

                if ri == 0:
                    _add_phase_date_label(ax, phase.upper(), date_txt, phase_colors.get(phase, '#888888'))

                if ci == 0:
                    ax.text(-0.24, 0.5, row_cfg['label'], transform=ax.transAxes, va='center', ha='center',
                            rotation=90, fontsize=8.2, fontweight='bold', color='#333333')

            # House cleaning 
            del raw_data, decomp, plot_dict
            gc.collect()

        # Allocate Colorbars to the entire side row tracking layout
        for ri, row_cfg in enumerate(SCALE_GRAD_ROWS):
            cfg = row_cfg[var.lower()]
            if row_ims[ri] is not None:
                cb = fig.colorbar(row_ims[ri], cax=cbar_axes[ri], orientation='vertical', extend='max')
                cb.set_label(cfg['unit'], fontsize=8.0)
                cb.ax.tick_params(labelsize=7.0)

        # Legends and Title Wrapping Context
        fig.legend(
            handles=[
                plt.Line2D([0], [0], color='white',  lw=0.8, linestyle='--', label='Full track'),
                plt.Line2D([0], [0], color='yellow', lw=1.2, marker='o', ms=3, label='Track to date'),
                plt.Line2D([0], [0], color='red',    lw=0,  marker='*', ms=8, label='Current position'),
            ],
            loc='lower right', bbox_to_anchor=(0.915, 0.055), fontsize=7.0, framealpha=0.85, edgecolor='#cccccc'
        )

        fig.suptitle(f"{cyc['name']}  --  {var} Gradient Scaled Matrix\n{window_label} {start_str} -> {end_str}", 
                     fontsize=14, fontweight='bold', y=0.98)

        if save:
            out = cyclone_out_dir or OUTPUT_DIR
            fname = f'{cyclone_name}_{var}_Gradient_{window_label}_4x{n_cols}.png'
            fpath = os.path.join(out, fname)
            fig.savefig(fpath, dpi=150, bbox_inches='tight', facecolor='white')
            print(f"Saved -> {fpath}")
            plt.close(fig)
            figures[window_label] = fpath
        else:
            figures[window_label] = fig
            
    gc.collect()
    return figures

# Helper block mapping multi-cyclone runtime execution exactly like standard matrix call:
def plot_scale_gradient_matrix(cyclone_name='ALL', variable='SST',
                                save=True, output_dir=None):
    '''
    Top-level runner for the gradient-magnitude evolution matrix.

    Parameters
    ----------
    cyclone_name : str or None, optional
        ALL (or None) for all cyclones, or a single name.  Default ALL.
    variable : str, optional
        SST or MSLA.  Default SST.
    save : bool, optional
        Save figures to disk.  Default True.
    output_dir : str or None, optional
    '''
    if cyclone_name == 'ALL' or cyclone_name is None:
        cyclone_names = [c for c, info in cyclones.items() if info.get('dates')]
        for name in cyclone_names:
            _plot_scale_gradient_matrix_one(name, variable=variable, save=save, output_dir=output_dir)
            plt.close('all')
    else:
        _plot_scale_gradient_matrix_one(cyclone_name, variable=variable, save=save, output_dir=output_dir)
        plt.close('all')


figs_sst_grads = plot_scale_gradient_matrix('ALL', variable='SST', save=True)
figs_msla_grads = plot_scale_gradient_matrix('ALL', variable='MSLA', save=True)



<!-- Plot sst and msla anomalies Across all the Cyclones And put them on a graph Where y axis is sst anomalies And X axis is Date of that cyclone also highlight the cyclone with some dots or square triangle all of these things so plot 2 curves one for sst and second for msla anomalies Then can you predict Correlation studies between Msla and sst And make one plot For all the Cyclones that how SST anomalies and MSLA anomaly across all the cyclones are correlated -->

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr


# =============================================================================
# STEP 1 — Compute simple mean SST & MSLA anomaly per cyclone
# =============================================================================

def _get_peak_date(cyc):
    """
    Returns a datetime.date for the cyclone peak.
    Priority:
      1. dates['peak']
      2. midpoint(formation, dissipation)
      3. midpoint of analysis window
    """
    dates = cyc.get('dates', {})
    if dates.get('peak'):
        return pd.to_datetime(dates['peak']).date()
    if dates.get('formation') and dates.get('dissipation'):
        t0 = pd.to_datetime(dates['formation'])
        t1 = pd.to_datetime(dates['dissipation'])
        return (t0 + (t1 - t0) / 2).date()
    windows = _get_analysis_window_ranges(cyc)
    if windows:
        _, s, e = windows[-1]
        t0, t1 = pd.to_datetime(s), pd.to_datetime(e)
        return (t0 + (t1 - t0) / 2).date()
    return None


def calculate_cyclone_anomalies_mean(cyclones_dict,
                                     output_csv='cyclone_anomalies_mean.csv'):
    '''
    Compute mean meso-scale SST and MSLA anomalies for every cyclone.

    Iterates over all dates in the analysis window, loads daily fields,
    applies scale decomposition, and averages the meso-scale band over
    the domain and full time window.

    Parameters
    ----------
    cyclones_dict : dict
    output_csv : str, optional

    Returns
    -------
    pd.DataFrame
        One row per cyclone: Cyclone, Peak_Date, Formation, Dissipation,
        SST_Anomaly, MSLA_Anomaly.
    '''
    records = []

    for cyc_name, cyc in cyclones_dict.items():
        print(f"  {cyc_name} ...", end=' ', flush=True)

        bbox  = cyc.get('bbox')
        track = cyc.get('track')
        if not bbox or not track:
            print("skipped (no bbox/track)"); continue

        bbox      = _make_square_bbox(bbox, track)
        peak_date = _get_peak_date(cyc)
        if peak_date is None:
            print("skipped (no peak date)"); continue

        windows = _get_analysis_window_ranges(cyc)
        if not windows:
            print("skipped (no analysis window)"); continue

        _, start_str, end_str = windows[-1]
        date_list = _get_analysis_date_list_from_range(start_str, end_str)

        sst_vals, msla_vals = [], []

        for date in date_list:
            # SST
            sst_raw = load_sst_for_date(date, bbox)
            if sst_raw is not None:
                dec = decompose_sst_field(sst_raw)
                if dec and 'meso' in dec:
                    sst_vals.append(np.nanmean(dec['meso']))

            # MSLA
            msla_raw = load_msla_for_date(date, bbox)
            if msla_raw is not None:
                dec = decompose_msla_field(msla_raw)
                if dec and 'meso' in dec:
                    msla_vals.append(np.nanmean(dec['meso']))

        sst_mean  = np.nanmean(sst_vals)  if sst_vals  else np.nan
        msla_mean = np.nanmean(msla_vals) if msla_vals else np.nan

        d = cyc.get('dates', {})
        records.append({
            'Cyclone'     : cyc_name,
            'Peak_Date'   : str(peak_date),
            'Formation'   : d.get('formation', ''),
            'Dissipation' : d.get('dissipation', ''),
            'SST_Anomaly' : sst_mean,
            'MSLA_Anomaly': msla_mean,
        })
        print(f"SST={sst_mean:.4f}  MSLA={msla_mean:.5f}")

    df = pd.DataFrame(records)
    if not df.empty:
        df['Peak_Date'] = pd.to_datetime(df['Peak_Date'])
        df.to_csv(output_csv, index=False)
        print(f"\nSaved → {output_csv}")
    return df


# =============================================================================
# STEP 2 — Plot SST Anomaly  (x = Peak Date, y = SST Anomaly)
# =============================================================================

def plot_sst_anomaly(df, save=True):
    '''
    Stem-chart of mean meso-scale SST anomaly per cyclone vs. peak date.

    Parameters
    ----------
    df : pd.DataFrame
    save : bool, optional  Default True.
    '''
    valid = df.dropna(subset=['SST_Anomaly', 'Peak_Date']).copy()
    valid['Peak_Date'] = pd.to_datetime(valid['Peak_Date'])
    valid = valid.sort_values('Peak_Date')

    if valid.empty:
        print("No SST data."); return

    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h',
               'H', 'X', 'd', 'P', '8']
    colors  = ['#d62728' if v >= 0 else '#1f77b4'
               for v in valid['SST_Anomaly']]

    fig, ax = plt.subplots(figsize=(14, 6))

    # zero line
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)

    # vertical stems
    for _, row in valid.iterrows():
        ax.plot([row['Peak_Date'], row['Peak_Date']],
                [0, row['SST_Anomaly']],
                color='#cccccc', linewidth=0.8, zorder=2)

    # markers + value labels
    for i, (_, row) in enumerate(valid.iterrows()):
        c = '#d62728' if row['SST_Anomaly'] >= 0 else '#1f77b4'
        ax.scatter(row['Peak_Date'], row['SST_Anomaly'],
                   marker=markers[i % len(markers)],
                   color=c, s=80, zorder=4,
                   edgecolors='white', linewidths=0.5)

        # cyclone name
        yoff = 0.004 if row['SST_Anomaly'] >= 0 else -0.004
        ax.annotate(row['Cyclone'],
                    xy=(row['Peak_Date'], row['SST_Anomaly']),
                    xytext=(0, 10 if row['SST_Anomaly'] >= 0 else -12),
                    textcoords='offset points',
                    ha='center', fontsize=6.5, color='#333333',
                    rotation=45,
                    arrowprops=dict(arrowstyle='-', color='#aaaaaa', lw=0.5))

        # numeric label
        # ax.text(row['Peak_Date'], row['SST_Anomaly'] + yoff * 8,
        #         f"{row['SST_Anomaly']:.3f}",
        #         ha='center', va='bottom' if row['SST_Anomaly'] >= 0 else 'top',
        #         fontsize=6, color='#444444')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    fig.autofmt_xdate(rotation=45)

    ax.set_xlabel('Peak Date', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean SST Anomaly (°C)', fontsize=12, fontweight='bold')
    ax.set_title('Mean Mesoscale SST Anomaly per Cyclone vs. Peak Date',
                 fontsize=14, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.25)

    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#d62728', label='Warm anomaly (+)'),
                        Patch(color='#1f77b4', label='Cold anomaly (−)')],
              fontsize=9, framealpha=0.9, loc='best')

    plt.tight_layout()
    if save:
        plt.savefig('SST_Anomaly_vs_Date.png', dpi=150, bbox_inches='tight')
        print("Saved: SST_Anomaly_vs_Date.png")
    plt.show()


# =============================================================================
# STEP 3 — Plot MSLA Anomaly  (x = Peak Date, y = MSLA Anomaly)
# =============================================================================

def plot_msla_anomaly(df, save=True):
    '''
    Stem-chart of mean meso-scale MSLA anomaly per cyclone vs. peak date.

    Parameters
    ----------
    df : pd.DataFrame
    save : bool, optional  Default True.
    '''
    valid = df.dropna(subset=['MSLA_Anomaly', 'Peak_Date']).copy()
    valid['Peak_Date'] = pd.to_datetime(valid['Peak_Date'])
    valid = valid.sort_values('Peak_Date')

    if valid.empty:
        print("No MSLA data."); return

    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h',
               'H', 'X', 'd', 'P', '8']

    fig, ax = plt.subplots(figsize=(14, 6))

    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)

    for _, row in valid.iterrows():
        ax.plot([row['Peak_Date'], row['Peak_Date']],
                [0, row['MSLA_Anomaly']],
                color='#cccccc', linewidth=0.8, zorder=2)

    for i, (_, row) in enumerate(valid.iterrows()):
        c = '#2ca02c' if row['MSLA_Anomaly'] >= 0 else '#9467bd'
        ax.scatter(row['Peak_Date'], row['MSLA_Anomaly'],
                   marker=markers[i % len(markers)],
                   color=c, s=80, zorder=4,
                   edgecolors='white', linewidths=0.5)

        ax.annotate(row['Cyclone'],
                    xy=(row['Peak_Date'], row['MSLA_Anomaly']),
                    xytext=(0, 10 if row['MSLA_Anomaly'] >= 0 else -12),
                    textcoords='offset points',
                    ha='center', fontsize=6.5, color='#333333',
                    rotation=45,
                    arrowprops=dict(arrowstyle='-', color='#aaaaaa', lw=0.5))

        yoff = 0.0003 if row['MSLA_Anomaly'] >= 0 else -0.0003
        # ax.text(row['Peak_Date'], row['MSLA_Anomaly'] + yoff * 8,
        #         f"{row['MSLA_Anomaly']:.4f}",
        #         ha='center', va='bottom' if row['MSLA_Anomaly'] >= 0 else 'top',
        #         fontsize=6, color='#444444')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    fig.autofmt_xdate(rotation=45)

    ax.set_xlabel('Peak Date', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean MSLA Anomaly (m)', fontsize=12, fontweight='bold')
    ax.set_title('Mean Mesoscale MSLA Anomaly per Cyclone vs. Peak Date',
                 fontsize=14, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.25)

    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#2ca02c', label='Positive MSLA (+)'),
                        Patch(color='#9467bd', label='Negative MSLA (−)')],
              fontsize=9, framealpha=0.9, loc='best')

    plt.tight_layout()
    if save:
        plt.savefig('MSLA_Anomaly_vs_Date.png', dpi=150, bbox_inches='tight')
        print("Saved: MSLA_Anomaly_vs_Date.png")
    plt.show()


# =============================================================================
# STEP 4 — Correlation  (x = SST, y = MSLA, one point per cyclone)
# =============================================================================

def plot_anomaly_correlation(df, save=True):
    '''
    Scatter plot of meso-scale SST vs MSLA anomaly across all cyclones.

    Parameters
    ----------
    df : pd.DataFrame
    save : bool, optional  Default True.
    '''
    # Note: Using column names from user's snippet. Ensure these exist in your DataFrame.
    # If using the multiscale CSV, you might need to rename columns or map them here.
    col_x = 'SST_Anomaly'
    col_y = 'MSLA_Anomaly'
    
    if col_x not in df.columns or col_y not in df.columns:
        # Fallback to Meso-scale if user's names aren't found
        col_x, col_y = 'SST_Meso', 'MSLA_Meso'

    valid = df.dropna(subset=[col_x, col_y]).copy()
    if valid.empty:
        print("No paired data."); return

    sst  = valid[col_x].values
    msla = valid[col_y].values
    corr, pval = pearsonr(sst, msla)

    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', 'h', 'H',
               'X', 'd', 'P', '8', '*']
    
    # Modern colormap lookup
    cmap = plt.get_cmap('tab20', len(valid))

    fig, ax = plt.subplots(figsize=(9, 7))

    # Calculate midpoints for intelligent annotation placement
    x_mid = np.mean(sst)
    y_mid = np.mean(msla)

    for i, (_, row) in enumerate(valid.iterrows()):
        x, y = row[col_x], row[col_y]
        ax.scatter(x, y,
                   marker=markers[i % len(markers)],
                   color=cmap(i), s=90, zorder=4,
                   edgecolors='white', linewidths=0.5,
                   label=row['Cyclone'])
        
        # --- START MODIFIED ANNOTATION LOGIC ---
        # Adjust text position based on location relative to height/width center
        # This prevents labels from going out of plot bounds.
        x_off = 5 if x < x_mid else -5
        y_off = 5 if y < y_mid else -5
        ha = 'left' if x_off > 0 else 'right'
        va = 'bottom' if y_off > 0 else 'top'

        ax.annotate(row['Cyclone'],
                    xy=(x, y),
                    xytext=(x_off, y_off), textcoords='offset points',
                    ha=ha, va=va,
                    fontsize=6.5, color='#333333',
                    bbox=dict(boxstyle='round,pad=0.1', facecolor='white', edgecolor='none', alpha=0.5))
        # --- END MODIFIED ANNOTATION LOGIC ---

    # regression
    m, b   = np.polyfit(sst, msla, 1)
    xs     = np.linspace(sst.min(), sst.max(), 200)
    ax.plot(xs, m * xs + b, color='crimson', linestyle='--',
            linewidth=1.8, label=f'Linear fit  r={corr:.2f},  p={pval:.3f}')

    ax.axhline(0, color='gray', linewidth=0.6)
    ax.axvline(0, color='gray', linewidth=0.6)

    ax.set_xlabel('Mean SST Anomaly (°C)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean MSLA Anomaly (m)',  fontsize=12, fontweight='bold')
    ax.set_title('SST ↔ MSLA Correlation — All Cyclones',
                 fontsize=13, fontweight='bold', pad=12)
    ax.grid(True, alpha=0.25)

    # legend outside to the right in two columns
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0),
              fontsize=6.5, framealpha=0.9, ncol=2, 
              title="Cyclones", title_fontsize=8,
              labelspacing=0.3, borderpad=0.5)

    ax.text(0.02, 0.97,
            f'Pearson r = {corr:.3f}\np-value   = {pval:.4f}\nn = {len(valid)}',
            transform=ax.transAxes, va='top', ha='left', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                      edgecolor='#cccccc', alpha=0.9))

    # Manual adjustment to prevent "Tight layout" warnings and accommodate the 2-column legend
    plt.subplots_adjust(left=0.1, right=0.7, top=0.9, bottom=0.15)
    if save:
        plt.savefig('SST_MSLA_Correlation.png', dpi=150, bbox_inches='tight')
        print("Saved: SST_MSLA_Correlation.png")
    plt.show()


# =============================================================================
# RUNNER  (uncomment to execute)
# =============================================================================

# -- compute once (slow) ---
# df_anom = calculate_cyclone_anomalies_mean(cyclones)

# -- or reload saved CSV --
# df_anom = pd.read_csv('cyclone_anomalies_mean.csv')

# # -- plot --
# plot_sst_anomaly(df_anom)
# plot_msla_anomaly(df_anom)
# plot_anomaly_correlation(df_anom)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os

def _bar_plot_abs_per_date(df, col, title, y_label, save_name, save=True, std_threshold=None):
    """
    Plots the absolute magnitude of the anomaly for each cyclone across the timeline.
    Colors bars red for original warm (positive) anomalies and blue for cold (negative).
    """
    if col not in df.columns:
        print(f"No data for {col}")
        return

    # Check if the corresponding std column exists
    std_col = f"{col}_Std"
    has_std = std_col in df.columns

    # Keep only rows with valid anomaly/std/date values
    if has_std:
        valid = df.dropna(subset=[col, std_col, 'Peak_Date']).copy()
    else:
        valid = df.dropna(subset=[col, 'Peak_Date']).copy()

    if valid.empty:
        print(f"No data for {col}")
        return

    # Ensure dates are datetime
    valid['Peak_Date'] = pd.to_datetime(valid['Peak_Date'])

    # Sort chronologically
    valid = valid.sort_values('Peak_Date')

    # Color bars
    colors = ['#d62728' if v >= 0 else '#1f77b4' for v in valid[col]]

    # Absolute values for plotting
    abs_vals = valid[col].abs()

    # Figure size
    fig_width = min(20, max(12, len(valid) * 0.4))
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    # Plot bars
    bars = ax.bar(
        valid['Peak_Date'],
        abs_vals,
        color=colors,
        edgecolor=None,
        linewidth=0,
        width=0.7
    )

    # Zero line
    ax.axhline(0, color='gray', linewidth=0.8, zorder=2)

    # Titles and labels
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel(y_label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Peak Date', fontsize=12, fontweight='bold')

    # Date formatting
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    if len(valid) <= 12:
        ax.xaxis.set_major_locator(mdates.MonthLocator())
    else:
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

    # Restrict timeline
    ax.set_xlim([
        pd.to_datetime('2023-12-15'),
        pd.to_datetime('2025-01-15')
    ])

    fig.autofmt_xdate(rotation=45)

    ax.grid(axis='y', alpha=0.3)

    # Offset for labels
    ymin, ymax = abs_vals.min(), abs_vals.max()
    offset = 0.02 * (ymax - ymin) if ymax != ymin else 0.01

    # Legend for Warm/Cold
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d62728', label='Warm (Original > 0)'),
        Patch(facecolor='#1f77b4', label='Cold (Original < 0)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')

    # ============================================================
    # STD THRESHOLD
    # ============================================================
    actual_std_threshold = (
        std_threshold
        if std_threshold is not None
        else valid[std_col].mean() if has_std else 0
    )

    # ============================================================
    # BAR ANNOTATIONS
    # ============================================================
    for i, (bar, name) in enumerate(zip(bars, valid['Cyclone'])):
        height = bar.get_height()

        # Standard deviation anomaly text
        std_text = ""
        if has_std:
            std_val = valid[std_col].iloc[i]
            if std_val > actual_std_threshold:
                std_text = f"\nΔσ={std_val:.3f}"

        annot_text = f"{name}{std_text}"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            annot_text,
            ha='right',
            va='bottom',
            fontsize=6,
            rotation=45,
            rotation_mode='anchor',
            color='#333333'
        )

    # ============================================================
    # STATISTICS BOX
    # ============================================================
    col_mean = valid[col].mean()
    col_std = valid[col].std()
    
    # We show the original data stats (mean and std) but can add abs mean
    abs_mean = abs_vals.mean()

    stats_text = (
        f"n = {len(valid)}\n"
        f"Original Mean = {col_mean:.3f}\n"
        f"Original Std = {col_std:.3f}\n"
        f"Abs Mean = {abs_mean:.3f}"
    )

    if has_std:
        stats_text += f"\nΔσ Threshold = {actual_std_threshold:.3f}"

    ax.text(
        0.02,
        0.95,
        stats_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment='top',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            alpha=0.8,
            edgecolor='#cccccc'
        )
    )

    # Layout adjustment
    plt.subplots_adjust(
        bottom=0.25,
        top=0.9,
        left=0.08,
        right=0.98
    )

    # Save figure
    if save:
        plt.savefig(save_name, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_name}")

    plt.close(fig)

def plot_submeso_abs(df):
    scales = [
        ('SST_Submeso', 'Sub-mesoscale SST Absolute Anomaly Magnitude', 'Absolute SST Anomaly (°C)', 0.10),
        ('MSLA_Submeso', 'Sub-mesoscale MSLA Absolute Anomaly Magnitude', 'Absolute MSLA Anomaly (m)', 0.05)
    ]
    for col, title, y_label, std_thresh in scales:
        _bar_plot_abs_per_date(df, col, title, y_label, f"{col}_Abs_vs_Date.png", save=True, std_threshold=std_thresh)

# if __name__ == '__main__':
#     csv_file = 'cyclone_multiscale_anomalies.csv'
#     if not os.path.exists(csv_file):
#         print(f"Error: {csv_file} not found.")
#     else:
#         df = pd.read_csv(csv_file)
#         plot_submeso_abs(df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os

def _bar_plot_abs_per_date(df, col, title, y_label, save_name, save=True, std_threshold=None):
    """
    Plots the absolute magnitude of the anomaly for each cyclone across the timeline.
    Colors bars red for original warm (positive) anomalies and blue for cold (negative).
    """
    if col not in df.columns:
        print(f"No data for {col}")
        return

    # Check if the corresponding std column exists
    std_col = f"{col}_Std"
    has_std = std_col in df.columns

    # Keep only rows with valid anomaly/std/date values
    if has_std:
        valid = df.dropna(subset=[col, std_col, 'Peak_Date']).copy()
    else:
        valid = df.dropna(subset=[col, 'Peak_Date']).copy()

    if valid.empty:
        print(f"No data for {col}")
        return

    # Ensure dates are datetime
    valid['Peak_Date'] = pd.to_datetime(valid['Peak_Date'])

    # Sort chronologically
    valid = valid.sort_values('Peak_Date')

    # Color bars
    colors = ['#d62728' if v >= 0 else '#1f77b4' for v in valid[col]]

    # Absolute values for plotting
    abs_vals = valid[col].abs()

    # Figure size
    fig_width = min(20, max(12, len(valid) * 0.4))
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    # Plot bars
    bars = ax.bar(
        valid['Peak_Date'],
        abs_vals,
        color=colors,
        edgecolor=None,
        linewidth=0,
        width=0.7
    )

    # Zero line
    ax.axhline(0, color='gray', linewidth=0.8, zorder=2)

    # Titles and labels
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel(y_label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Peak Date', fontsize=12, fontweight='bold')

    # Date formatting
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    if len(valid) <= 12:
        ax.xaxis.set_major_locator(mdates.MonthLocator())
    else:
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

    # Restrict timeline
    ax.set_xlim([
        pd.to_datetime('2023-12-15'),
        pd.to_datetime('2025-01-15')
    ])

    fig.autofmt_xdate(rotation=45)

    ax.grid(axis='y', alpha=0.3)

    # Offset for labels
    ymin, ymax = abs_vals.min(), abs_vals.max()
    offset = 0.02 * (ymax - ymin) if ymax != ymin else 0.01

    # Legend for Warm/Cold
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d62728', label='Warm (Original > 0)'),
        Patch(facecolor='#1f77b4', label='Cold (Original < 0)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')

    # ============================================================
    # STD THRESHOLD
    # ============================================================
    actual_std_threshold = (
        std_threshold
        if std_threshold is not None
        else valid[std_col].mean() if has_std else 0
    )

    # ============================================================
    # BAR ANNOTATIONS
    # ============================================================
    for i, (bar, name) in enumerate(zip(bars, valid['Cyclone'])):
        height = bar.get_height()

        # Standard deviation anomaly text
        std_text = ""
        if has_std:
            std_val = valid[std_col].iloc[i]
            if std_val > actual_std_threshold:
                std_text = f"\nΔσ={std_val:.3f}"

        annot_text = f"{name}{std_text}"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            annot_text,
            ha='right',
            va='bottom',
            fontsize=6,
            rotation=45,
            rotation_mode='anchor',
            color='#333333'
        )

    # ============================================================
    # STATISTICS BOX
    # ============================================================
    col_mean = valid[col].mean()
    col_std = valid[col].std()
    
    # We show the original data stats (mean and std) but can add abs mean
    abs_mean = abs_vals.mean()

    stats_text = (
        f"n = {len(valid)}\n"
        f"Original Mean = {col_mean:.3f}\n"
        f"Original Std = {col_std:.3f}\n"
        f"Abs Mean = {abs_mean:.3f}"
    )

    if has_std:
        stats_text += f"\nΔσ Threshold = {actual_std_threshold:.3f}"

    ax.text(
        0.02,
        0.95,
        stats_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment='top',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            alpha=0.8,
            edgecolor='#cccccc'
        )
    )

    # Layout adjustment
    plt.subplots_adjust(
        bottom=0.25,
        top=0.9,
        left=0.08,
        right=0.98
    )

    # Save figure
    if save:
        plt.savefig(save_name, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_name}")
    plt.show()
    plt.close(fig)

def plot_submeso_abs(df):
    scales = [
        ('SST_Submeso', 'Sub-mesoscale SST Absolute Anomaly Magnitude', 'Absolute SST Anomaly (°C)', 0.10),
        ('MSLA_Submeso', 'Sub-mesoscale MSLA Absolute Anomaly Magnitude', 'Absolute MSLA Anomaly (m)', 0.05)
    ]
    for col, title, y_label, std_thresh in scales:
        _bar_plot_abs_per_date(df, col, title, y_label, f"{col}_Abs_vs_Date.png", save=True, std_threshold=std_thresh)

if __name__ == '__main__':
    csv_file = 'cyclone_multiscale_anomalies.csv'
    if not os.path.exists(csv_file):
        print(f"Error: {csv_file} not found.")
    else:
        df = pd.read_csv(csv_file)
        plot_submeso_abs(df)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

print("Calculating absolute raw means over analysis windows for all cyclones (this may take a minute)...\n")

raw_records = []

for cyc_name, cyc in cyclones.items():
    print(f"  {cyc_name}...", end=" ")
    bbox = _make_square_bbox(cyc.get('bbox', {}), cyc.get('track', []))
    peak_date = _get_peak_date(cyc)
    
    if not peak_date:
        print("Skipped (no peak date)")
        continue
        
    windows = _get_analysis_window_ranges(cyc)
    if not windows:
        print("Skipped (no window)")
        continue
        
    _, start_str, end_str = windows[-1]
    date_list = _get_analysis_date_list_from_range(start_str, end_str)

    sst_vals, msla_vals = [], []
    for date in date_list:
        # Load raw data
        sst_raw = load_sst_for_date(date, bbox)
        if sst_raw is not None:
            sst_vals.append(np.nanmean(sst_raw))
            
        msla_raw = load_msla_for_date(date, bbox)
        if msla_raw is not None:
            msla_vals.append(np.nanmean(msla_raw))
            
    sst_mean = np.nanmean(sst_vals) if sst_vals else np.nan
    msla_mean = np.nanmean(msla_vals) if msla_vals else np.nan
    
    raw_records.append({
        'Cyclone': cyc_name,
        'Peak_Date': str(peak_date),
        'SST_Raw': sst_mean,
        'MSLA_Raw': msla_mean
    })
    print("Done")

df_raw = pd.DataFrame(raw_records)
df_raw['Peak_Date'] = pd.to_datetime(df_raw['Peak_Date'])

print("\nFinished building raw dataset!")


# =============================================================================
# Plotting function mimicking `plot_msla_anomaly` exactly
# =============================================================================

def plot_raw_stem_chart(df, column, ylabel, title, save_name, is_msla=False):
    valid = df.dropna(subset=[column, 'Peak_Date']).copy()
    valid = valid.sort_values('Peak_Date')

    if valid.empty:
        print(f"No data for {column}.")
        return

    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'H', 'X', 'd', 'P', '8']

    fig, ax = plt.subplots(figsize=(14, 6))

    # Because absolute temperatures are ~28C (not zero), we draw the baseline
    # at the overall global average of the dataset so stems go up and down.
    overall_mean = valid[column].mean()
    ax.axhline(overall_mean, color='gray', linewidth=0.8, linestyle='--', zorder=1, label=f'Annual Baseline ({overall_mean:.2f})')

    for _, row in valid.iterrows():
        ax.plot([row['Peak_Date'], row['Peak_Date']],
                [overall_mean, row[column]],
                color='#cccccc', linewidth=0.8, zorder=2)

    for i, (_, row) in enumerate(valid.iterrows()):
        
        # Color logic: green/purple for MSLA, red/blue for SST based on relation to the mean
        if is_msla:
            c = '#2ca02c' if row[column] >= overall_mean else '#9467bd'
            yoff = 0.0003
        else:
            c = '#d62728' if row[column] >= overall_mean else '#1f77b4'
            yoff = 0.004
            
        ax.scatter(row['Peak_Date'], row[column],
                   marker=markers[i % len(markers)],
                   color=c, s=80, zorder=4,
                   edgecolors='white', linewidths=0.5)

        direct_yoff = 10 if row[column] >= overall_mean else -12
        ax.annotate(row['Cyclone'],
                    xy=(row['Peak_Date'], row[column]),
                    xytext=(0, direct_yoff),
                    textcoords='offset points',
                    ha='center', fontsize=6.5, color='#333333',
                    rotation=45,
                    arrowprops=dict(arrowstyle='-', color='#aaaaaa', lw=0.5))

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    fig.autofmt_xdate(rotation=45)

    ax.set_xlabel('Peak Date', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.25)
    
    if is_msla:
         handles = [Patch(color='#2ca02c', label='Above Global Mean (+)'),
                    Patch(color='#9467bd', label='Below Global Mean (−)')]
    else:
         handles = [Patch(color='#d62728', label='Warm Background (+)'),
                    Patch(color='#1f77b4', label='Cold Background (−)')]
         
    ax.legend(handles=handles, fontsize=9, framealpha=0.9, loc='best')

    plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()

# =============================================================================
# Run Plots!
# =============================================================================
# plot_raw_stem_chart(df_raw, 'SST_Raw', 'Absolute Raw SST (°C)', 'Mean Absolute Raw SST per Cyclone vs. Peak Date', 'Raw_SST_vs_Date.png', is_msla=False)
# plot_raw_stem_chart(df_raw, 'MSLA_Raw', 'Absolute Raw MSLA (m)', 'Mean Absolute Raw MSLA per Cyclone vs. Peak Date', 'Raw_MSLA_vs_Date.png', is_msla=True)
# Insert this below the 'Run Plots!' section in cell 29

def plot_raw_correlation(df, save_name='Raw_Correlation_SST_vs_MSLA.png'):
    valid = df.dropna(subset=['SST_Raw', 'MSLA_Raw', 'Cyclone']).copy()
    if valid.empty:
         print("No valid data to correlate.")
         return
         
    # 1. Z-Score normalization
    valid['SST_Z'] = (valid['SST_Raw'] - valid['SST_Raw'].mean()) / valid['SST_Raw'].std()
    valid['MSLA_Z'] = (valid['MSLA_Raw'] - valid['MSLA_Raw'].mean()) / valid['MSLA_Raw'].std()

    # 2. Setup Figure
    fig, ax = plt.subplots(figsize=(9, 8))
    
    # 3. Compute Pearson
    r, p = pearsonr(valid['SST_Z'], valid['MSLA_Z'])
    
    # 4. Scatter Plot
    ax.scatter(valid['SST_Z'], valid['MSLA_Z'], 
               color='#d62728', edgecolor='white', s=100, zorder=3, alpha=0.85)

    # 5. Trendline
    m, b = np.polyfit(valid['SST_Z'], valid['MSLA_Z'], 1)
    x_range = np.linspace(valid['SST_Z'].min()-0.5, valid['SST_Z'].max()+0.5, 100)
    ax.plot(x_range, m*x_range + b, color='#333333', linewidth=2, linestyle='--',
            label=f'Trend (r = {r:.2f}, p = {p:.3f})', zorder=2)
            
    # 6. Annotations (Cyclone Names)
    for i, row in valid.iterrows():
       ax.annotate(row['Cyclone'],
                   (row['SST_Z'], row['MSLA_Z']),
                   textcoords="offset points",
                   xytext=(5, 5),
                   ha='left',
                   fontsize=8,
                   alpha=0.8)

    # 7. Formatting
    ax.axhline(0, color='gray', linestyle='-', linewidth=0.5, zorder=1)
    ax.axvline(0, color='gray', linestyle='-', linewidth=0.5, zorder=1)
    
    ax.set_xlabel('Normalized Raw SST (Z-Score)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Normalized Raw MSLA (Z-Score)', fontsize=12, fontweight='bold')
    ax.set_title('Normalized Correlation: Absolute Raw SST vs. Absolute Raw MSLA', fontsize=14, fontweight='bold', pad=15)
    
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()

# Run the correlation plot
# plot_raw_correlation(df_raw, save_name='Raw_Correlation_SST_vs_MSLA.png')




In [ ]:

def plot_raw_correlation_absolute(df, save_name='Raw_Correlation_Absolute_SST_vs_MSLA.png'):
    '''
    Scatter plot of absolute (non-normalised) raw SST vs MSLA per cyclone.

    Parameters
    ----------
    df : pd.DataFrame
        Columns: SST_Raw, MSLA_Raw, Cyclone.
    save_name : str, optional
    '''
    valid = df.dropna(subset=['SST_Raw', 'MSLA_Raw', 'Cyclone']).copy()
    if valid.empty:
         print("No valid data to correlate.")
         return
         
    # Setup Figure
    fig, ax = plt.subplots(figsize=(9, 8))
    
    # Compute Pearson Correlation on absolute values
    r, p = pearsonr(valid['SST_Raw'], valid['MSLA_Raw'])
    
    # Scatter Plot
    ax.scatter(valid['SST_Raw'], valid['MSLA_Raw'], 
               color='#ff7f0e', edgecolor='white', s=100, zorder=3, alpha=0.85)

    # Trendline
    m, b = np.polyfit(valid['SST_Raw'], valid['MSLA_Raw'], 1)
    
    # Create x-range slightly wider than data to draw trendline nicely
    x_min, x_max = valid['SST_Raw'].min(), valid['SST_Raw'].max()
    x_range = np.linspace(x_min - 0.2, x_max + 0.2, 100)
    
    ax.plot(x_range, m * x_range + b, color='#333333', linewidth=2, linestyle='--',
            label=f'Trend (r = {r:.2f}, p = {p:.3f})', zorder=2)
            
    # Annotations (Cyclone Names)
    for i, row in valid.iterrows():
       ax.annotate(row['Cyclone'],
                   (row['SST_Raw'], row['MSLA_Raw']),
                   textcoords="offset points",
                   xytext=(5, 5),
                   ha='left',
                   fontsize=8,
                   alpha=0.8)

    # Formatting
    ax.set_xlabel('Absolute Raw SST (°C)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Absolute Raw MSLA (m)', fontsize=12, fontweight='bold')
    ax.set_title('Correlation: Absolute Raw SST vs. Absolute Raw MSLA (Unnormalized)', fontsize=14, fontweight='bold', pad=15)
    
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()

# Run the absolute correlation plot
# plot_raw_correlation_absolute(df_raw, save_name='Raw_Correlation_Absolute_SST_vs_MSLA.png')

In [ ]:
import pandas as pd
import numpy as np
import json
import os
# from cyclone_anomaly_analysis import (
#     _get_peak_date, _make_square_bbox, _get_analysis_window_ranges,
#     _get_analysis_date_list_from_range, load_sst_for_date, decompose_sst_field
# )

def calculate_sst_submeso_gradients(cyclones_dict):
    '''
    Compute the mean sub-mesoscale SST gradient magnitude for every cyclone.

    Parameters
    ----------
    cyclones_dict : dict

    Returns
    -------
    pd.DataFrame
        Columns: Cyclone, SST_Submeso_Grad_Avg (deg C / km).
    '''
    records = []
    for cyc_name, cyc in cyclones_dict.items():
        print(f"Processing {cyc_name} ...", end=' ', flush=True)

        bbox = cyc.get('bbox')
        track = cyc.get('track')
        if not bbox or not track:
            print("skipped (no bbox/track)")
            continue

        bbox = _make_square_bbox(bbox, track)
        peak_date = _get_peak_date(cyc)
        if peak_date is None:
            print("skipped (no peak date)")
            continue

        windows = _get_analysis_window_ranges(cyc)
        if not windows:
            print("skipped (no analysis window)")
            continue

        # Target Signal (from During/Post window)
        _, start_str, end_str = windows[-1]
        date_list = _get_analysis_date_list_from_range(start_str, end_str)

        sst_sub_grad_vals = []
        ref_lat = (bbox['lat_min'] + bbox['lat_max']) / 2.0
        # SST grid resolution is 0.05 degrees
        dy_km = 111.0 * 0.05
        dx_km = 111.0 * np.cos(np.deg2rad(ref_lat)) * 0.05

        for date in date_list:
            sst_raw = load_sst_for_date(date, bbox)
            if sst_raw is not None:
                dec = decompose_sst_field(sst_raw)
                if dec and 'submeso' in dec:
                    submeso_field = dec['submeso']
                    # Compute gradient (axis 0 = lat/y, axis 1 = lon/x)
                    grad_y, grad_x = np.gradient(submeso_field)
                    
                    # Convert to physical units: degrees Celsius per km
                    grad_y_km = grad_y / dy_km
                    grad_x_km = grad_x / dx_km
                    
                    # Magnitude of the gradient
                    grad_mag = np.sqrt(grad_x_km**2 + grad_y_km**2)
                    
                    # Average over the region
                    mean_grad = np.nanmean(grad_mag)
                    sst_sub_grad_vals.append(mean_grad)

        if sst_sub_grad_vals:
            # Average over the dates
            mean_grad_for_cyc = np.nanmean(sst_sub_grad_vals)
            rec = {
                'Cyclone': cyc_name,
                'SST_Submeso_Grad_Avg': mean_grad_for_cyc
            }
            print(f"Grad={mean_grad_for_cyc:.5f} °C/km")
            records.append(rec)
        else:
            print("No data.")

    return pd.DataFrame(records)

# if __name__ == "__main__":
#     json_path = 'cyclone_info_2024_padded.json'
#     csv_file = 'cyclone_multiscale_anomalies.csv'
    
#     if os.path.exists(json_path) and os.path.exists(csv_file):
#         with open(json_path, 'r') as f:
#             raw_data = json.load(f)
            
#         CYCLONES = {}
#         for basin, b_data in raw_data.items():
#             for storm in b_data.get('storms', []):
#                 name = storm.get('name', '').upper()
#                 CYCLONES[name] = storm
            
#         grad_df = calculate_sst_submeso_gradients(CYCLONES)
        
#         # Merge with existing CSV
#         df = pd.read_csv(csv_file)
#         if 'SST_Submeso_Grad_Avg' in df.columns:
#             df = df.drop(columns=['SST_Submeso_Grad_Avg'])
            
#         df = pd.merge(df, grad_df, on='Cyclone', how='left')
#         df.to_csv(csv_file, index=False)
#         print(f"\nAppended SST_Submeso_Grad_Avg to {csv_file}")
#     else:
#         print("Missing json or csv file.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os

def plot_gradient_timeline(df, col, title, y_label, save_name):
    '''
    Bar chart of gradient magnitude per cyclone ordered by peak date.

    Parameters
    ----------
    df : pd.DataFrame
    col : str  Column containing gradient magnitude values.
    title : str
    y_label : str
    save_name : str  Output filename (PNG).
    '''
    valid = df.dropna(subset=[col, 'Peak_Date']).copy()
    if valid.empty:
        print(f"No data for {col}")
        return
        
    valid['Peak_Date'] = pd.to_datetime(valid['Peak_Date'])
    valid = valid.sort_values('Peak_Date')

    # Figure size
    fig_width = min(20, max(12, len(valid) * 0.4))
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    # All values are magnitudes (>=0), using a distinct color
    color = '#2ca02c' # Green for spatial gradient

    bars = ax.bar(
        valid['Peak_Date'],
        valid[col],
        color=color,
        edgecolor=None,
        linewidth=0,
        width=0.7
    )

    ax.axhline(0, color='gray', linewidth=0.8, zorder=2)

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel(y_label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Peak Date', fontsize=12, fontweight='bold')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    if len(valid) <= 12:
        ax.xaxis.set_major_locator(mdates.MonthLocator())
    else:
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

    ax.set_xlim([
        pd.to_datetime('2023-12-15'),
        pd.to_datetime('2025-01-15')
    ])

    fig.autofmt_xdate(rotation=45)
    ax.grid(axis='y', alpha=0.3)

    ymin, ymax = valid[col].min(), valid[col].max()
    offset = 0.02 * (ymax - ymin) if ymax != ymin else 0.01

    for bar, name in zip(bars, valid['Cyclone']):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            name,
            ha='right',
            va='bottom',
            fontsize=6,
            rotation=45,
            rotation_mode='anchor',
            color='#333333'
        )

    # Statistics box
    col_mean = valid[col].mean()
    col_std = valid[col].std()

    stats_text = (
        f"n = {len(valid)}\n"
        f"Mean = {col_mean:.5f}\n"
        f"Std Dev = {col_std:.5f}"
    )

    ax.text(
        0.02,
        0.95,
        stats_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment='top',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            alpha=0.8,
            edgecolor='#cccccc'
        )
    )

    plt.subplots_adjust(
        bottom=0.25,
        top=0.9,
        left=0.08,
        right=0.98
    )

    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    print(f"Saved: {save_name}")
    plt.show()
    plt.close(fig)
    
# if __name__ == '__main__':
#     csv_file = 'cyclone_multiscale_anomalies.csv'
#     if not os.path.exists(csv_file):
#         print(f"Error: {csv_file} not found.")
#     else:
#         df = pd.read_csv(csv_file)
#         plot_gradient_timeline(
#             df,
#             col='SST_Submeso_Grad_Avg',
#             title='Sub-mesoscale SST Average Spatial Gradient Magnitude',
#             y_label='Gradient Magnitude (°C/km)',
#             save_name='SST_Submeso_Grad_Avg_vs_Date.png'
#         )


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import json

# =============================================================================
# STEP 0 — LOAD INTENSITY DATA FROM JSON
# =============================================================================

def load_intensity_data(json_path='cyclone_info_2024_padded.json'):
    '''
    Extract intensity metadata for all storms from the cyclone JSON.

    Returns
    -------
    pd.DataFrame
        Columns: Cyclone, Category, Max_Wind_mph, Min_Pressure_mb.
    '''
    with open(json_path, 'r') as f:
        data = json.load(f)

    records = []
    for basin_key, basin in data.items():
        if not isinstance(basin, dict) or 'storms' not in basin:
            continue
        for s in basin['storms']:
            records.append({
                'Cyclone': s['name'],
                'Category': s.get('category', ''),
                'Max_Wind_mph': s.get('max_wind_mph', np.nan),
                'Min_Pressure_mb': s.get('min_pressure_mb', np.nan),
            })
    return pd.DataFrame(records)


# =============================================================================
# STEP 1 — MERGE ANOMALIES + INTENSITY
# =============================================================================

def build_coci_dataframe(anomaly_csv='cyclone_multiscale_anomalies.csv',
                         json_path='cyclone_info_2024_padded.json'):
    '''
    Build the merged DataFrame used for all COCI formulations.

    Reads the anomaly CSV, loads intensity data from the JSON, merges on
    cyclone name, and computes z-score normalisations for SST, MSLA, and
    max wind speed.

    Returns
    -------
    pd.DataFrame
        Merged and z-score normalised DataFrame ready for compute_all_indices.
    '''
    df_anom = pd.read_csv(anomaly_csv)
    df_int  = load_intensity_data(json_path)

    # normalise names to uppercase so CSV and JSON always match
    df_anom['Cyclone'] = df_anom['Cyclone'].str.upper()
    df_int['Cyclone']  = df_int['Cyclone'].str.upper()

    # merge on cyclone name
    df = df_anom.merge(df_int, on='Cyclone', how='left')

    # use meso-scale as primary anomaly (most relevant for cyclone coupling)
    df = df.rename(columns={'SST_Meso': 'SST_Anom', 'MSLA_Meso': 'MSLA_Anom'})

    # drop rows missing critical columns
    df = df.dropna(subset=['SST_Anom', 'MSLA_Anom', 'Max_Wind_mph'])

    # z-score normalisation
    for col, zcol in [('SST_Anom', 'Z_SST'),
                      ('MSLA_Anom', 'Z_MSLA'),
                      ('Max_Wind_mph', 'Z_V')]:
        mu, sigma = df[col].mean(), df[col].std()
        df[zcol] = (df[col] - mu) / sigma if sigma > 0 else 0.0

    return df


# =============================================================================
# STEP 2 — COMPUTE ALL INDEX VARIANTS
# =============================================================================

def compute_all_indices(df):
    '''
    Compute all four COCI index variants and append them as new columns.

    Variants: COCI_A (equal-weight), COCI_B (multiplicative),
    COCI_C (correlation-weighted ocean), COCI_D (ocean-only average).

    Returns
    -------
    pd.DataFrame
        Input DataFrame with added columns COCI_A, COCI_B, COCI_C, COCI_D.
    '''

    # --- Option A: Equal-weight linear ---
    df['COCI_A'] = (df['Z_SST'] + df['Z_MSLA'] + df['Z_V']) / 3.0

    # --- Option B: Ocean forcing × intensity (multiplicative) ---
    df['COCI_B'] = (df['Z_SST'] + df['Z_MSLA']) * df['Z_V']

    # --- Option C: Correlation-weighted (data-driven) ---
    if len(df) >= 2:
        r_sst, _  = pearsonr(df['SST_Anom'], df['Max_Wind_mph'])
        r_msla, _ = pearsonr(df['MSLA_Anom'], df['Max_Wind_mph'])
    else:
        r_sst, r_msla = 0.5, 0.5
    w_sst  = abs(r_sst)
    w_msla = abs(r_msla)
    w_sum  = w_sst + w_msla
    if w_sum > 0:
        df['COCI_C'] = (w_sst * df['Z_SST'] + w_msla * df['Z_MSLA']) / w_sum
    else:
        df['COCI_C'] = (df['Z_SST'] + df['Z_MSLA']) / 2.0

    # --- Option D: Simple ocean-only (no intensity) ---
    df['COCI_D'] = (df['Z_SST'] + df['Z_MSLA']) / 2.0

    # store the correlation weights for reporting
    df.attrs['r_sst']  = r_sst
    df.attrs['r_msla'] = r_msla

    return df


# =============================================================================
# STEP 3 — CLASSIFY EACH CYCLONE
# =============================================================================

def classify_coci(value):
    if value > 2:
        return 'Extreme'
    elif value > 1:
        return 'Strong'
    elif value > 0:
        return 'Moderate'
    elif value > -1:
        return 'Weak'
    else:
        return 'Suppressed'


def add_classifications(df):
    for idx_col in ['COCI_A', 'COCI_B', 'COCI_C', 'COCI_D']:
        df[f'{idx_col}_Class'] = df[idx_col].apply(classify_coci)
    return df


# =============================================================================
# STEP 4 — VALIDATE: correlate each index with Vmax
# =============================================================================

def validate_indices(df):
    '''
    Compute Pearson correlation between each COCI variant and max wind speed.

    Returns
    -------
    pd.DataFrame
        One row per index: Index, Pearson_r, p_value, Mean, Std.
    '''
    results = []
    for idx_col, label in [
        ('COCI_A', 'A: Equal-Weight Linear'),
        ('COCI_B', 'B: Ocean × Intensity'),
        ('COCI_C', 'C: Correlation-Weighted'),
        ('COCI_D', 'D: Ocean-Only'),
    ]:
        if len(df) >= 2:
            r, p = pearsonr(df[idx_col], df['Max_Wind_mph'])
        else:
            r, p = 0.0, 1.0
        results.append({
            'Index': label,
            'Column': idx_col,
            'Pearson_r': r,
            'p_value': p,
            'Mean': df[idx_col].mean(),
            'Std': df[idx_col].std(),
        })
    return pd.DataFrame(results)


# =============================================================================
# STEP 5 — VALIDATE: mean COCI by cyclone category
# =============================================================================

def category_validation(df):
    '''
    Compute mean COCI by cyclone category (TS, 1, 2, 3, 4, 5).

    Returns
    -------
    pd.DataFrame
        Index = category; columns = COCI_A through COCI_D plus Count.
    '''
    cat_order = ['TS', '1', '2', '3', '4', '5']
    idx_cols = ['COCI_A', 'COCI_B', 'COCI_C', 'COCI_D']
    grp = df.groupby('Category')[idx_cols].mean()
    grp = grp.reindex([c for c in cat_order if c in grp.index])
    grp['Count'] = df.groupby('Category')['Cyclone'].count()
    return grp


# =============================================================================
# PLOTS
# =============================================================================

def plot_index_comparison(df, save=True):
    '''
    Grouped bar chart comparing all four COCI index variants per cyclone.

    Cyclones sorted by ascending max wind speed.

    Parameters
    ----------
    df : pd.DataFrame
    save : bool, optional  Default True.
    '''
    idx_cols = ['COCI_A', 'COCI_B', 'COCI_C', 'COCI_D']
    labels   = ['A: Equal-Wt', 'B: Ocean×V', 'C: Corr-Wt', 'D: Ocean-Only']
    colors   = ['#5C6BC0', '#26A69A', '#FFA726', '#EF5350']

    df_sorted = df.sort_values('Max_Wind_mph')
    x = np.arange(len(df_sorted))
    width = 0.2

    fig, ax = plt.subplots(figsize=(max(14, len(df_sorted) * 0.6), 6))
    for i, (col, lab, clr) in enumerate(zip(idx_cols, labels, colors)):
        ax.bar(x + i * width, df_sorted[col].values, width,
               label=lab, color=clr, edgecolor='white', zorder=3)

    ax.set_xticks(x + 1.5 * width)
    ax.set_xticklabels(df_sorted['Cyclone'], rotation=60, fontsize=7, ha='right')
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set_ylabel('COCI Value', fontsize=12, fontweight='bold')
    ax.set_title('COCI Comparison — All Variants (sorted by wind speed)',
                 fontsize=13, fontweight='bold', pad=10)
    ax.legend(fontsize=8, framealpha=0.9)
    ax.grid(axis='y', alpha=0.25, zorder=0)
    plt.tight_layout()
    if save:
        plt.savefig('COCI_Comparison.png', dpi=150, bbox_inches='tight')
        print("Saved: COCI_Comparison.png")
    plt.show()


def plot_best_index_scatter(df, best_col, best_label, save=True):
    '''
    Scatter plot of the best-performing COCI variant vs max wind speed.

    Parameters
    ----------
    df : pd.DataFrame
    best_col : str
    best_label : str
    save : bool, optional  Default True.
    '''
    r, p = pearsonr(df[best_col], df['Max_Wind_mph'])

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(df[best_col], df['Max_Wind_mph'],
                    c=df['Max_Wind_mph'], cmap='YlOrRd',
                    s=80, edgecolors='white', linewidths=0.5, zorder=4)
    # annotate
    for _, row in df.iterrows():
        ax.annotate(row['Cyclone'],
                    xy=(row[best_col], row['Max_Wind_mph']),
                    xytext=(4, 4), textcoords='offset points',
                    fontsize=5.5, color='#444444')
    # regression
    m, b = np.polyfit(df[best_col].values, df['Max_Wind_mph'].values, 1)
    xs = np.linspace(df[best_col].min(), df[best_col].max(), 200)
    ax.plot(xs, m * xs + b, 'k--', lw=1.5,
            label=f'Fit  r={r:.3f},  p={p:.4f}')

    ax.set_xlabel(f'{best_label}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Max Wind (mph)', fontsize=12, fontweight='bold')
    ax.set_title(f'Best Index Validation: {best_label} vs Intensity',
                 fontsize=13, fontweight='bold', pad=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)
    plt.colorbar(sc, ax=ax, label='Max Wind (mph)')
    plt.tight_layout()
    if save:
        plt.savefig('COCI_Best_Scatter.png', dpi=150, bbox_inches='tight')
        print("Saved: COCI_Best_Scatter.png")
    plt.show()


def plot_category_means(cat_df, save=True):
    '''
    Grouped bar chart of mean COCI value by Saffir-Simpson category.

    Parameters
    ----------
    cat_df : pd.DataFrame  Output of category_validation.
    save : bool, optional  Default True.
    '''
    idx_cols = ['COCI_A', 'COCI_B', 'COCI_C', 'COCI_D']
    labels   = ['A: Equal-Wt', 'B: Ocean×V', 'C: Corr-Wt', 'D: Ocean-Only']
    colors   = ['#5C6BC0', '#26A69A', '#FFA726', '#EF5350']

    x = np.arange(len(cat_df))
    width = 0.2

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (col, lab, clr) in enumerate(zip(idx_cols, labels, colors)):
        vals = cat_df[col].fillna(0).values
        ax.bar(x + i * width, vals, width,
               label=lab, color=clr, edgecolor='white', zorder=3)

    ax.set_xticks(x + 1.5 * width)
    ax.set_xticklabels([f"Cat {c}\n(n={int(cat_df.loc[c, 'Count'])})"
                        for c in cat_df.index], fontsize=9)
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set_ylabel('Mean COCI', fontsize=12, fontweight='bold')
    ax.set_title('Mean COCI by Cyclone Category',
                 fontsize=14, fontweight='bold', pad=10)
    ax.legend(fontsize=8, framealpha=0.9)
    ax.grid(axis='y', alpha=0.25, zorder=0)
    plt.tight_layout()
    if save:
        plt.savefig('COCI_Category_Means.png', dpi=150, bbox_inches='tight')
        print("Saved: COCI_Category_Means.png")
    plt.show()


# =============================================================================
# MASTER RUNNER
# =============================================================================

def run_coci_analysis(anomaly_csv='cyclone_multiscale_anomalies.csv',
                      json_path='cyclone_info_2024_padded.json',
                      save=True):
    '''
    Run the complete Cyclone Ocean Coupling Index (COCI) pipeline.

    Steps: build_coci_dataframe -> compute_all_indices -> add_classifications
    -> validate_indices -> category_validation -> plots -> CSV export.

    Returns
    -------
    tuple
        (df, val_df, cat_df, best_col)
    '''

    print("=" * 60)
    print("  CYCLONE OCEAN COUPLING INDEX (COCI) ANALYSIS")
    print("=" * 60)

    # 1. Build merged dataframe
    df = build_coci_dataframe(anomaly_csv, json_path)
    print(f"\nMerged {len(df)} cyclones with SST/MSLA anomalies + intensity.\n")

    # 2. Compute all indices
    df = compute_all_indices(df)
    df = add_classifications(df)

    # 3. Validate
    val_df = validate_indices(df)
    print("─── Validation: Pearson r (Index vs Max Wind) ───")
    print(val_df[['Index', 'Pearson_r', 'p_value']].to_string(index=False))

    # 4. Identify best
    best_row = val_df.loc[val_df['Pearson_r'].abs().idxmax()]
    best_col = best_row['Column']
    best_label = best_row['Index']
    print(f"\n★ BEST INDEX: {best_label}  (r = {best_row['Pearson_r']:.4f})")

    # 5. Category validation
    cat_df = category_validation(df)
    print("\n─── Mean COCI by Category ───")
    print(cat_df.to_string())

    # 6. Correlation weights used in Option C
    print(f"\nCorrelation weights (Option C):")
    print(f"  corr(SST, Vmax)  = {df.attrs.get('r_sst', 0):.4f}")
    print(f"  corr(MSLA, Vmax) = {df.attrs.get('r_msla', 0):.4f}")

    # 7. Plots
    plot_index_comparison(df, save)
    plot_best_index_scatter(df, best_col, best_label, save)
    plot_category_means(cat_df, save)

    # 8. Save full table
    out_csv = 'coci_results.csv'
    df.to_csv(out_csv, index=False)
    print(f"\nFull results → {out_csv}")

    val_csv = 'coci_validation.csv'
    val_df.to_csv(val_csv, index=False)
    print(f"Validation   → {val_csv}")

    return df, val_df, cat_df, best_col


# =============================================================================
# RUN  (uncomment to execute)
# =============================================================================
# df_coci, val, cat, best = run_coci_analysis()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_and_annotate_high_std(csv_file='cyclone_multiscale_anomalies.csv', std_column='SST_Submeso_Std', threshold=0.1):
    """
    Plots the Standard Deviation for all cyclones and annotates the ones 
    that cross a specific threshold value.
    """
    # Load the exported anomaly data
    df = pd.read_csv(csv_file)
    
    # Drop rows without the specific std data
    df = df.dropna(subset=[std_column])
    
    # Sort for better visual representation
    df = df.sort_values(by=std_column).reset_index(drop=True)
    
    plt.figure(figsize=(14, 7))
    
    # Setup x indexing
    x_positions = np.arange(len(df))
    
    # Bar plot for all cyclones
    bars = plt.bar(x_positions, df[std_column], color='#90CAF9', edgecolor='black')
    
    # Draw threshold line
    plt.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold})')
    
    # Find and annotate outliers
    outliers_found = False
    for i, row in df.iterrows():
        val = row[std_column]
        if val > threshold:
            outliers_found = True
            # Color the bar red if it exceeds threshold
            bars[i].set_color('#EF5350')
            bars[i].set_edgecolor('black')
            
            # Annotate the cyclone name & value above the bar
            plt.text(x_positions[i], val + (val * 0.02), 
                     f"{row['Cyclone']}\n({val:.3f})", 
                     ha='center', va='bottom', fontsize=9, fontweight='bold', color='darkred', rotation=45)
            
    plt.xticks(x_positions, df['Cyclone'], rotation=90, fontsize=8)
    plt.ylabel(f'{std_column.replace("_", " ")} Value', fontsize=12, fontweight='bold')
    plt.title(f'Cyclones Exceeding {std_column} Threshold (> {threshold})', fontsize=14, fontweight='bold', pad=15)
    
    if outliers_found:
        plt.legend()
    else:
        plt.title(f'No Cyclones Exceeded {std_column} Threshold (> {threshold})', fontsize=14, fontweight='bold', pad=15)
        
    plt.margins(y=0.15) # Add padding at the top for labels
    plt.tight_layout()
    plt.show()

# ==========================================
# Run the function (adjust target and threshold as needed)
# ==========================================

# # Example 1: Finding Submeso SST Standard Deviation Outliers > 0.15
# plot_and_annotate_high_std(std_column='SST_Submeso_Std', threshold=0.15)

# # Example 2: Finding Meso MSLA Standard Deviation Outliers > 0.05
# plot_and_annotate_high_std(std_column='MSLA_Meso_Std', threshold=0.05)

In [ ]:
# =============================================================================
# STEP 6 — NORMALIZE BEST COCI TO 0-1 SCALE & PLOT RANKING
# =============================================================================

def calculate_coci_0_to_1(df, best_col):
    """
    Applies Min-Max normalization to the best COCI variant.
    Maps everything to a strict [0, 1] range.
    """
    min_val = df[best_col].min()
    max_val = df[best_col].max()
    
    # Mathematical scaling to 0 - 1
    df['COCI_Scale_0_to_1'] = (df[best_col] - min_val) / (max_val - min_val)
    
    # Creating an alternative 0 to 100 percentage score (often easier to read)
    df['COCI_Score_100'] = df['COCI_Scale_0_to_1'] * 100
    
    return df

def plot_coci_scores(df, best_col, save=True):
    """Bar chart ranking all cyclones from 1.0 (Highest) to 0.0 (Lowest)."""
    # Sort cyclones by their new score so the graph acts as a leaderboard
    df_sorted = df.sort_values('COCI_Scale_0_to_1', ascending=False)
    
    fig, ax = plt.subplots(figsize=(15, 6))
    
    # Use a colormap so high numbers are yellow/green, low numbers are dark purple
    colors = plt.cm.viridis(df_sorted['COCI_Scale_0_to_1'])
    
    bars = ax.bar(df_sorted['Cyclone'], df_sorted['COCI_Scale_0_to_1'], 
                  color=colors, edgecolor='white', linewidth=1)
    
    ax.set_ylabel('Ocean-Cyclone Coupling Score (0 to 1)', fontsize=12, fontweight='bold')
    ax.set_title(f'Cyclone Ranking based on Normalized Best COCI ({best_col})', fontsize=14, fontweight='bold', pad=15)
    
    ax.set_xticklabels(df_sorted['Cyclone'], rotation=60, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.15) # Give room for labels on top
    
    # Annotate the specific 0.xx value on top of each bar
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.2f}', 
                ha='center', va='bottom', fontsize=8, rotation=90, color='#333333')
        
    plt.tight_layout()
    if save:
        plt.savefig('COCI_Score_0to1_Ranking.png', dpi=150)
        print("Saved: COCI_Score_0to1_Ranking.png")
    plt.show()

# --- Run the function using the 'best' variable exported from your master runner ---
# df_coci = calculate_coci_0_to_1(df_coci, best)

# print("\n─── Top 5 Cyclones by 0-1 Coupling Score ───")
# print(df_coci[['Cyclone', 'Category', best, 'COCI_Scale_0_to_1', 'COCI_Score_100']].sort_values('COCI_Scale_0_to_1', ascending=False).head())

# plot_coci_scores(df_coci, best)

# Save updated dataset with the new 0_val_1 scoring columns
# df_coci.to_csv('coci_results_with_scoring.csv', index=False)

In [ ]:
def plot_individual_coci_components(df, save=True):
    """
    Grouped bar chart comparing the individual normalized components 
    (Z_SST, Z_MSLA, Z_V) for each cyclone to see their individual effects.
    Sorted by maximum wind speed (Z_V).
    """
    if 'Z_SST' not in df.columns or df.empty:
        print("Required normalized columns (Z_SST, Z_MSLA, Z_V) not found.")
        return
        
    # Sort by normalized wind speed to see how ocean components react to intensity
    df_sorted = df.sort_values('Z_V')
    
    x = np.arange(len(df_sorted))
    width = 0.25  # standard width for 3 side-by-side bars
    
    fig, ax = plt.subplots(figsize=(max(14, len(df_sorted) * 0.5), 6))
    
    # Plot the 3 components
    ax.bar(x - width, df_sorted['Z_SST'], width, 
           label='SST Anomaly (Z_SST)', color='#1f77b4', edgecolor='white', alpha=0.9)
           
    ax.bar(x, df_sorted['Z_MSLA'], width, 
           label='MSLA Anomaly (Z_MSLA)', color='#2ca02c', edgecolor='white', alpha=0.9)
           
    ax.bar(x + width, df_sorted['Z_V'], width, 
           label='Max Wind (Z_V)', color='#d62728', edgecolor='white', alpha=0.9)

    ax.set_xticks(x)
    ax.set_xticklabels(df_sorted['Cyclone'], rotation=60, ha='right', fontsize=8.5)
    
    ax.axhline(0, color='gray', linewidth=1, zorder=1)
    
    ax.set_ylabel('Normalized Score (Z-Score / Std Dev)', fontsize=12, fontweight='bold')
    ax.set_title('Individual Component Comparison: SST vs. MSLA vs. Wind Intensity', 
                 fontsize=14, fontweight='bold', pad=15)
                 
    ax.legend(fontsize=10, loc='best')
    ax.grid(axis='y', alpha=0.25)
    
    plt.tight_layout()
    if save:
        plt.savefig('COCI_Individual_Components.png', dpi=150, bbox_inches='tight')
        print("Saved: COCI_Individual_Components.png")
    plt.show()

# =============================================================================
# Add this line to the bottom of your MASTER RUNNER function (run_coci_analysis)
# OR simply run it alone like this if your dataframe is already loaded:
# =============================================================================
# plot_individual_coci_components(df_coci, save=True)

In [ ]:
# =============================================================================
# INDIVIDUAL COMPONENT RANKINGS (0 to 1 SCALE)
# =============================================================================

def calculate_individual_0_to_1(df):
    """
    Normalizes Max Wind, SST Anomaly, and MSLA Anomaly to a 0-1 scale.
    Note: SST and MSLA are inverted so that the MOST NEGATIVE anomaly (strongest cooling/depression) gets 1.0.
    """
    # Wind: Higher is stronger
    min_v, max_v = df['Max_Wind_mph'].min(), df['Max_Wind_mph'].max()
    df['Wind_Scale_0_to_1'] = (df['Max_Wind_mph'] - min_v) / (max_v - min_v)
    
    # SST: More negative is stronger
    min_sst, max_sst = df['SST_Anom'].min(), df['SST_Anom'].max()
    df['SST_Scale_0_to_1'] = (max_sst - df['SST_Anom']) / (max_sst - min_sst)
    
    # MSLA: More negative is stronger
    min_msla, max_msla = df['MSLA_Anom'].min(), df['MSLA_Anom'].max()
    df['MSLA_Scale_0_to_1'] = (max_msla - df['MSLA_Anom']) / (max_msla - min_msla)
    
    return df

def plot_single_ranking(df, col_name, title, ylabel, save_name, save=True):
    """Helper to plot a single leaderboard."""
    df_sorted = df.sort_values(col_name, ascending=False)
    
    fig, ax = plt.subplots(figsize=(15, 6))
    colors = plt.cm.plasma(df_sorted[col_name]) # Using plasma for a distinct look
    
    bars = ax.bar(df_sorted['Cyclone'], df_sorted[col_name], 
                  color=colors, edgecolor='white', linewidth=1)
    
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    
    ax.set_xticklabels(df_sorted['Cyclone'], rotation=60, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.15)
    
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.2f}', 
                ha='center', va='bottom', fontsize=8, rotation=90, color='#333333')
        
    plt.tight_layout()
    if save:
        plt.savefig(save_name, dpi=150)
        print(f"Saved: {save_name}")
    plt.show()

def plot_all_individual_rankings(df):
    """Generates the three individual ranking plots."""
    df = calculate_individual_0_to_1(df)
    
    plot_single_ranking(df, 'Wind_Scale_0_to_1', 
                        'Cyclone Ranking based on Max Wind Speed (0 to 1)', 
                        'Wind Intensity Score', 'Ranking_Wind_0to1.png')
                        
    plot_single_ranking(df, 'SST_Scale_0_to_1', 
                        'Cyclone Ranking based on SST Cooling Intensity (0 to 1)', 
                        'SST Cooling Score', 'Ranking_SST_0to1.png')
                        
    plot_single_ranking(df, 'MSLA_Scale_0_to_1', 
                        'Cyclone Ranking based on MSLA Depression Intensity (0 to 1)', 
                        'MSLA Depression Score', 'Ranking_MSLA_0to1.png')
                        
    return df

# --- RUN IT ---
# df_coci = plot_all_individual_rankings(df_coci)

In [ ]:
def plot_coci_a_dominance(df, save=True):
    """
    Calculates and plots the relative 'dominance' of each Z-score component 
    (Wind, SST, MSLA) in forming the cyclone's extremity.
    Uses absolute magnitudes to avoid negative sign cancellation.
    """
    if not all(c in df.columns for c in ['Z_SST', 'Z_MSLA', 'Z_V', 'COCI_A']):
        print("Required Z-score columns missing.")
        return
        
    df_dom = df.copy()
    
    # 1. Calculate absolute magnitudes (how 'extreme' the factor is, regardless of direction)
    df_dom['Abs_Z_SST'] = df_dom['Z_SST'].abs()
    df_dom['Abs_Z_MSLA'] = df_dom['Z_MSLA'].abs()
    df_dom['Abs_Z_V'] = df_dom['Z_V'].abs()
    
    # 2. Calculate Total Extremity
    df_dom['Total_Abs_Z'] = df_dom['Abs_Z_SST'] + df_dom['Abs_Z_MSLA'] + df_dom['Abs_Z_V']
    
    # 3. Calculate Percentage Dominance (0 to 100%)
    df_dom['Pct_V'] = (df_dom['Abs_Z_V'] / df_dom['Total_Abs_Z']) * 100
    df_dom['Pct_SST'] = (df_dom['Abs_Z_SST'] / df_dom['Total_Abs_Z']) * 100
    df_dom['Pct_MSLA'] = (df_dom['Abs_Z_MSLA'] / df_dom['Total_Abs_Z']) * 100
    
    # Sort the chart by the actual COCI_A score (highest to lowest)
    df_dom = df_dom.sort_values('COCI_A', ascending=False)
    
    # 4. Plot 100% Stacked Bar Chart
    fig, ax = plt.subplots(figsize=(15, 7))
    
    bars1 = ax.bar(df_dom['Cyclone'], df_dom['Pct_V'], 
                   label='Wind Dominance (Z_V)', color='#d62728', edgecolor='white', alpha=0.9)
    bars2 = ax.bar(df_dom['Cyclone'], df_dom['Pct_SST'], bottom=df_dom['Pct_V'], 
                   label='SST Dominance (Z_SST)', color='#1f77b4', edgecolor='white', alpha=0.9)
    bars3 = ax.bar(df_dom['Cyclone'], df_dom['Pct_MSLA'], bottom=df_dom['Pct_V'] + df_dom['Pct_SST'], 
                   label='MSLA Dominance (Z_MSLA)', color='#2ca02c', edgecolor='white', alpha=0.9)
    
    # Add a reference line for perfectly equal weighting
    ax.axhline(33.33, color='black', linestyle='--', alpha=0.8, linewidth=1.5, label='Equal Weight (33.3%)')
    
    ax.set_ylabel('Component Dominance (%)', fontsize=12, fontweight='bold')
    ax.set_title('Composition of Cyclone Extremity (Z-Score Relative Magnitudes)', 
                 fontsize=14, fontweight='bold', pad=15)
                 
    ax.set_xticklabels(df_dom['Cyclone'], rotation=60, ha='right', fontsize=9)
    ax.set_ylim(0, 100)
    
    # Legend formatting
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1), fontsize=10)
    
    plt.tight_layout()
    if save:
        plt.savefig('COCI_A_Dominance_Stacked.png', dpi=150, bbox_inches='tight')
        print("Saved: COCI_A_Dominance_Stacked.png")
    plt.show()
    
    return df_dom

# =============================================================================
# RUN THE FUNCTION
# =============================================================================
# df_dominance = plot_coci_a_dominance(df_coci, save=True)

In [ ]:
import json

# =============================================================================
# REGIONAL DOMINANCE ANALYSIS
# =============================================================================

def add_basin_info(df, json_path='cyclone_info_2024_padded.json'):
    '''
    Add an ocean basin column to a COCI results DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        COCI DataFrame with a Cyclone column (uppercase names).
    json_path : str, optional

    Returns
    -------
    pd.DataFrame
        Copy of df with an additional Basin column.
    '''
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    basin_map = {}
    for basin_key, basin_data in data.items():
        if not isinstance(basin_data, dict) or 'storms' not in basin_data:
            continue
        for s in basin_data['storms']:
            basin_map[s['name'].upper()] = basin_key.upper()
            
    df_out = df.copy()
    df_out['Basin'] = df_out['Cyclone'].map(basin_map)
    return df_out

def plot_regional_dominance(df, json_path='cyclone_info_2024_padded.json', save=True):
    """
    Groups cyclones by Basin and calculates which Z-score component 
    (Wind, SST, MSLA) dominates the COCI score on average for that region.
    """
    # 1. Attach basin info if missing
    if 'Basin' not in df.columns:
        df = add_basin_info(df, json_path)
        
    # 2. Re-calculate dominance mathematically cleanly
    df['Abs_Z_SST'] = df['Z_SST'].abs()
    df['Abs_Z_MSLA'] = df['Z_MSLA'].abs()
    df['Abs_Z_V'] = df['Z_V'].abs()
    
    df['Total_Abs_Z'] = df['Abs_Z_SST'] + df['Abs_Z_MSLA'] + df['Abs_Z_V']
    
    df['Pct_V'] = (df['Abs_Z_V'] / df['Total_Abs_Z']) * 100
    df['Pct_SST'] = (df['Abs_Z_SST'] / df['Total_Abs_Z']) * 100
    df['Pct_MSLA'] = (df['Abs_Z_MSLA'] / df['Total_Abs_Z']) * 100
    
    # 3. Filter valid rows
    valid = df.dropna(subset=['Pct_V', 'Pct_SST', 'Pct_MSLA', 'Basin'])
    if valid.empty:
        print("No valid basin data found.")
        return
        
    # 4. Group by Basin and get the averages
    regional_means = valid.groupby('Basin')[['Pct_V', 'Pct_SST', 'Pct_MSLA']].mean()
    cyclone_counts = valid.groupby('Basin')['Cyclone'].count()
    
    # Determine the statically dominating factor per region
    def get_dominant(row):
        labels = {'Pct_V': 'Wind', 'Pct_SST': 'SST', 'Pct_MSLA': 'MSLA'}
        return labels[row.idxmax()]
        
    regional_means['Dominant_Factor'] = regional_means.apply(get_dominant, axis=1)
    
    # Print the textual leaderboard
    print("\n" + "="*50)
    print("   REGIONAL COMPONENTS DOMINANCE SUMMARY")
    print("="*50)
    for basin in regional_means.index:
        count = cyclone_counts[basin]
        dom = regional_means.loc[basin, 'Dominant_Factor']
        v = regional_means.loc[basin, 'Pct_V']
        s = regional_means.loc[basin, 'Pct_SST']
        m = regional_means.loc[basin, 'Pct_MSLA']
        print(f"[{basin}] (n={count} cyclones)")
        print(f"  → Dominant Factor: {dom}")
        print(f"  → Wind: {v:.1f}% | SST: {s:.1f}% | MSLA: {m:.1f}%\n")
        
    # 5. Plotting a 100% stacked bar chart by Region
    x = np.arange(len(regional_means))
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Stack the bars
    ax.bar(x, regional_means['Pct_V'], 
           label='Wind Dominance (Z_V)', color='#d62728', edgecolor='white', width=0.6)
           
    ax.bar(x, regional_means['Pct_SST'], bottom=regional_means['Pct_V'], 
           label='SST Dominance (Z_SST)', color='#1f77b4', edgecolor='white', width=0.6)
           
    ax.bar(x, regional_means['Pct_MSLA'], bottom=regional_means['Pct_V'] + regional_means['Pct_SST'], 
           label='MSLA Dominance (Z_MSLA)', color='#2ca02c', edgecolor='white', width=0.6)
    
    ax.set_xticks(x)
    labels = [f"{b}\n(n={cyclone_counts[b]})" for b in regional_means.index]
    ax.set_xticklabels(labels, fontsize=5, fontweight='bold')
    
    # Draw reference line at exactly a 33% split
    ax.axhline(33.33, color='black', linestyle='--', alpha=0.8, linewidth=1.5, label='Equal Weight (33.3%)')
    
    ax.set_ylabel('Average Component Dominance (%)', fontsize=12, fontweight='bold')
    ax.set_title('Regional Comparison: Which Factor Dominates the Extremity Score?', 
                 fontsize=14, fontweight='bold', pad=15)
    
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
    
    plt.tight_layout()
    if save:
        plt.savefig('Regional_Component_Dominance.png', dpi=150, bbox_inches='tight')
        print(f"Saved: Regional_Component_Dominance.png")
    plt.show()

    return regional_means

# =============================================================================
# RUN THE REGIONAL FUNCTION
# =============================================================================
# df_regional_summary = plot_regional_dominance(df_coci, save=True) 8 minute ee jittebi function announce